# Geospatial Situation Awareness & the Perceived Opportunity Landscape
### Secondary-data trend analysis and basic time-series forecasting — Nepal & South Asia
### *V7 — revised for direct alignment with the paper's claims, theory, and figure/table needs*

This notebook builds a **country-year panel** from open, verifiable secondary sources,
integrates real **Asian Barometer Survey** individual-level microdata, and runs
descriptive, co-movement, forecasting, and mediation analyses on them, to produce
figures and results for the paper *"Spatial Cognition, Public Trust, and Youth
Migration."*

**Region:** Nepal (focal case) + India, Bangladesh, Pakistan, Sri Lanka, Bhutan —
the same five South Asian comparators used in the paper's Section 6.

**What changed in V7, in one line:** every section below now carries a markdown
cell stating *where it belongs in the paper*, *suggested paper text*, and *what it
infers* — and the underlying analysis was re-pointed at the paper's own constructs
(the "exit" variable, "democratic resilience," institution-level trust) instead of
generic placeholders, using data already loaded in V6. No new data source was added.
Three things were cut as out of scope (see Section 0b) rather than stretched to
cover something they can't.

**Run order:** Run cells top to bottom (`Runtime > Run all`), uploading your WGI
export, V-Dem CSV, and both ABS wave files exactly as before. This version has not
been re-executed since the rewrite — every figure/number below needs a fresh run
with your real files before anything is quoted in the paper.

## ⚠️ What this notebook can and cannot demonstrate

This is worth stating precisely, because it determines how these figures should be
described in the paper.

**What it *can* do:** pull real, citable, country-year series for governance quality,
remittances/migration, and socio-economic conditions; integrate real individual-level
survey data (Asian Barometer) for the trust/mediation side; show whether they trend
the way the paper's narrative says they do; and produce short-horizon statistical
forecasts of those *observed* series.

**What it *cannot* do:** test or verify the paper's actual theoretical contribution in
full. The *Perceived Opportunity Landscape* is explicitly defined as an individual-level
cognitive construct — perception → comprehension → projection of one's own
surroundings. Country-year aggregates (WGI, V-Dem, WDI) measure expert perceptions of
*countries*, not citizens' mental models of *their own* opportunity structure — that
gap is what the ABS-based mediation section (21) partially closes, and even there,
what gets tested is narrower than the full theory (see Section 0b and 21c).

Use these outputs to **support the empirical backdrop** (Section 2 / Section 5
"objective conditions" side of the paper), to supply the one real **individual-level
mediation test** the paper can currently make (Section 21), and as **motivation for
the future survey-based Geospatial Opportunity Accessibility Index** the paper
proposes in Section 8 — not as a complete quantitative test of Definition 1.1 or
Fig. 2 in one shot.

## 0b · Paper alignment map — where every section below fits, and what's deliberately out of scope

Read this once before anything else. It replaces guessing "does this notebook cover
RQ2?" with a direct answer.

| Notebook section | Feeds paper section / figure / table | Fig. 2 node it operationalizes |
|---|---|---|
| §5–14 (WGI, WDI, V-Dem trends: Figures A, A2, B, C) | Section 2 (sourced replacement/supplement for Fig. 1), Section 6 backdrop | Infrastructure quality (partial), Brain drain ↔ circulation, feedback via remittances |
| §12c (NEW — `v2x_regime` figure) | Section 5.7 / Table 1 groundwork | **Democratic resilience** (the model's own terminal node) |
| §15–16 (co-movement, retargeted) | Section 3.2 (Exit-Voice-Loyalty), Section 5.3 | **Migration aspirations (exit)** |
| §17–20 (forecasts) | Section 5.1 / 5.4 ("remittance economy masks stagnation") | Brain drain ↔ circulation |
| §21g–21i (ABS trust, aggregate + institution-level) | Section 5.1 / 5.6 ("crisis of trust") | **Public trust / institutional legitimacy** |
| §21j (ABS perceived condition) | Definition 1.1 | Closest available proxy for **Perceived Opportunity Landscape** |
| §21m (GBC32) | Section 3.3 (Endsley) | The **"projection"** stage specifically |
| §21n (mediation: Nepal + pooled, two outcomes) | Section 3.3/3.4, Section 8's "consistent with" language | Condition → Trust → **Voice/loyalty-adjacent outcome** |
| §21o (triangulation) | Section 3.3 (why POL is a distinct construct worth introducing) | — |

**Explicitly out of scope — said plainly, not implied away:**

- **RQ2 (spatial/institutional inequality between peripheral and core areas).** Every
  data source in this notebook (WGI, WDI, V-Dem, ABS) is a *national* aggregate or
  *national* survey sample — none has a subnational/district unit. No indicator swap
  within these four sources can speak to core-periphery dynamics. This needs SHDI
  and/or Nepal DoFE district data (Section 9), which are not loaded here. **Do not
  caption any figure below as evidence for RQ2.**
- **Brain circulation** (return migration, diaspora investment, re-engagement). None
  of the four sources contain a return-migration or diaspora-investment item.
  Remittances (§13) are re-captioned below as evidence of *loyalty* (continued
  financial engagement), which is related but is not evidence of *circulation*
  (people or capital coming back). **Do not caption the remittances figure as
  evidence of brain circulation.**
- **Literal migration/emigration intention** at the individual level. Confirmed absent
  from both ABS questionnaires (Section 21c). The mediation section's outcome is
  regime preference / satisfaction with democracy — a real, defensible, but
  narrower claim than "migration intent."

## 1 · Finalized data sources

| # | Category (as requested) | What we actually pull | Why |
|---|---|---|---|
| 1 | Trust & governance (Gallup / V-Dem / Asian Barometer / WVS) | **World Bank Worldwide Governance Indicators (WGI)**, loaded from a **manually downloaded file** — Voice & Accountability, Control of Corruption, Government Effectiveness, etc. | Gallup World Poll is subscription-only; WVS doesn't consistently field in Nepal; Asian Barometer covers Nepal but is registration-gated, not an API (see §9) — though its *individual-level microdata* is now loaded directly in §21, see below. V-Dem is free but bulk-file only (also §9). WGI itself is free with no registration either way — see the correction below for why this notebook loads it from a file you download once, rather than an API call. |
| 2 | Migration (UN DESA / IOM) | **World Bank mirror of UN DESA** (classic v2 API): net migration (`SM.POP.NETM`), migrant stock (`SM.POP.TOTL`), **+ personal remittances** (`BX.TRF.PWKR.CD.DT`, annual) | UN DESA's own release is 5-year-spaced Excel. IOM's portal visualizes UN DESA/UNICEF data rather than serving a bulk API. (V7: the direct UN DESA cross-check attempt in §8 was removed — it never resolved into a usable number, and `SM.POP.TOTL` is already UN-DESA-sourced via the WDI mirror, so it added an unfinished step without a distinct finding.) |
| 3 | Socio-economic (DHS / LSMS / World Bank) | **World Bank WDI** (classic v2 API): GDP per capita, secondary net enrollment, youth unemployment | DHS/LSMS are registration-gated microdata built for sub-national analysis, not quick country-year pulls — and sub-national is exactly what RQ2 would need; see §0b. |
| 4 | Governance & institutional quality (WGI) | **WGI, all six dimensions**, same manual file as row 1 | Exactly as requested. |

**Countries (ISO3):** `NPL` Nepal · `IND` India · `BGD` Bangladesh · `PAK` Pakistan · `LKA` Sri Lanka · `BTN` Bhutan

### ⚠️ Why WGI is a one-time manual download here, not an API call

Two rounds of live testing established: the classic `api.worldbank.org/v2` endpoint no
longer serves WGI's `.EST` indicators (confirmed directly — it returns *"the indicator
was not found... may have been deleted or archived"*), and WGI's replacement home, the
World Bank's new **Data360** platform, is in Beta and was intermittently slow/unreachable
during testing. WDI indicators (remittances, GDP, enrollment, unemployment, migration) are
unaffected and keep using the classic API below.

1. **Recommended — DataBank (lets you pre-filter to just these 6 countries):**
   `https://databank.worldbank.org/source/worldwide-governance-indicators`
   → Country: tick Nepal, India, Bangladesh, Pakistan, Sri Lanka, Bhutan
   → Series: tick the six **"...: Estimate"** series
   → Time: select all available years → **Download > CSV**
2. **Alternative — full dataset, one click:**
   `https://info.worldbank.org/governance/wgi/Home/downLoadFile?fileName=wgidataset.xlsx`

### V-Dem — trimmed in V7 to what the paper actually uses

V-Dem is still loaded (Section 5b/5c) as an enrichment of the *objective/institutional*
side, not a substitute for individual-level trust — same caveat as WGI applies to it.
**V7 removes three groups that V6 loaded but the paper never draws on**, purely to keep
the configuration honest about what's actually feeding the paper (this is a config
change only — nothing downstream needs to change because of it):

- **Cut: economic controls** (`e_gdppc`, `e_pop`, `e_miurbani`, `e_pematmor`,
  `e_civil_war`). `e_gdppc` duplicated WDI's `NY.GDP.PCAP.CD` already in this panel;
  the other four never map onto anything the paper argues.
- **Cut down: state capacity**, from three variables to one. Kept `v2strenadm`
  (administrative reach) as the closest proxy for Fig. 2's "institutional
  infrastructure quality" node; dropped fiscal capacity and territorial authority,
  which the paper doesn't discuss as distinct constructs.
- **Cut down: corruption**, from three variables to one (`v2x_corr`, overall). The
  paper never distinguishes public-sector from executive corruption, so the two
  branch-specific sub-indices were dropped.

**Kept, and now put to direct use in V7** (previously loaded but under-used — see §15
and §12c): `v2clfmove` — *the literal "exit" option in Hirschman's Exit-Voice-Loyalty
framework* — and `v2x_regime` — *V-Dem's own "Regimes of the World" classification,
the direct operationalization of "democratic resilience,"* Fig. 2's terminal node.
Also kept: the full democracy group (liberal democracy, civil liberties, civil
society — general governance-from-a-different-lens cross-check to WGI), the social
media-regulation battery (speaks directly to the Sept. 2025 crisis, Section 5.6/5.7),
and the WGI-bundled cross-check columns (internal QA only, Section 7b — not a
paper-facing result, so it isn't itemized in the table above).

In [ ]:
# 2 · Setup
!pip -q install statsmodels==0.14.* requests openpyxl semopy --upgrade

import os, time, json, warnings
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import statsmodels.api as sm
from statsmodels.tsa.holtwinters import Holt
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import grangercausalitytests

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)

OUT_DIR = "/content/outputs"
FIG_DIR = f"{OUT_DIR}/figures"
DATA_DIR = f"{OUT_DIR}/data"
for d in (OUT_DIR, FIG_DIR, DATA_DIR):
    os.makedirs(d, exist_ok=True)

print("Setup complete. Figures ->", FIG_DIR, "| Data ->", DATA_DIR)

In [ ]:
# 3 · Configuration: countries, indicators, years
COUNTRIES = ["NPL", "IND", "BGD", "PAK", "LKA", "BTN"]
FOCAL = "NPL"
COUNTRY_NAMES = {"NPL": "Nepal", "IND": "India", "BGD": "Bangladesh",
                 "PAK": "Pakistan", "LKA": "Sri Lanka", "BTN": "Bhutan"}

# Category 1 & 4 -- Worldwide Governance Indicators, loaded from a manually
# downloaded file. Matched by NAME inside load_wgi_manual(), so these keys
# are just internal labels.
WGI_INDICATORS = {
    "GOV_WGI_VA": "Voice & Accountability",
    "GOV_WGI_PV": "Political Stability & Absence of Violence",
    "GOV_WGI_GE": "Government Effectiveness",
    "GOV_WGI_RQ": "Regulatory Quality",
    "GOV_WGI_RL": "Rule of Law",
    "GOV_WGI_CC": "Control of Corruption",
}

# ---- V-Dem, V7: trimmed to groups with a nameable tie to the paper (see the
# markdown cell above for exactly what was cut and why) ----
VDEM_DEMOCRACY_INDICATORS = {
    "v2x_polyarchy": "Electoral democracy index (V-Dem)",
    "v2x_libdem": "Liberal democracy index (V-Dem)",
    "v2x_civlib": "Civil liberties index (V-Dem)",
    "v2xcs_ccsi": "Core civil society index (V-Dem)",
}
# v2x_regime is the paper's own stated operationalization of "democratic
# resilience" (Fig. 2's terminal node) -- given its own dedicated figure in
# Section 12c below rather than being one line among many.
VDEM_RESILIENCE_OUTCOME_INDICATORS = {
    "v2x_regime": "Democratic Resilience -- Regimes of the World, 0=Closed Autocracy..3=Liberal Democracy (V-Dem)",
}
# Trimmed from 3 to 1 (V7): overall corruption only -- the paper never
# distinguishes corruption by branch of government.
VDEM_CORRUPTION_INDICATORS = {
    "v2x_corr": "Political Corruption Index, overall (V-Dem)",
}
# v2clfmove is the star of this group -- "the literal exit option" -- now
# actually tested against migration/remittance outcomes in Section 15/16
# (it was loaded but never correlated with anything in V6).
VDEM_VOICE_EXIT_INDICATORS = {
    "v2clfmove": "Freedom of Foreign Movement (V-Dem) -- the 'exit' option in Hirschman's EVL framework",
    "v2x_freexp_altinf": "Freedom of Expression & Alternative Information (V-Dem)",
    "v2x_frassoc_thick": "Freedom of Association, thick (V-Dem)",
}
# Trimmed from 3 to 1 (V7): administrative reach only, as the closest proxy
# for Fig. 2's "institutional infrastructure quality" node.
VDEM_STATECAPACITY_INDICATORS = {
    "v2strenadm": "State Administrative Capacity / Reach (V-Dem)",
}
# Speaks directly to the Sept. 2025 social-media-ban episode (Section 5.6/5.7).
VDEM_SOCIALMEDIA_INDICATORS = {
    "v2smgovshutcap": "Government Social Media Shutdown Capacity (V-Dem)",
    "v2smgovshut": "Government Social Media Shutdown in Practice (V-Dem)",
    "v2smgovfilcap": "Government Internet Filtering Capacity (V-Dem)",
    "v2smregcap": "Government Social Media Regulation Capacity (V-Dem)",
}
# Internal QA only (Section 7b): confirms the manually-downloaded WGI file
# parsed correctly, by comparing it to V-Dem's own bundled WGI estimates.
# Not itemized as a paper-facing indicator group.
VDEM_WGI_CROSSCHECK_INDICATORS = {
    "e_wbgi_gee": "Government Effectiveness, WGI (bundled in V-Dem, QA only)",
    "e_wbgi_cce": "Control of Corruption, WGI (bundled in V-Dem, QA only)",
    "e_wbgi_rqe": "Regulatory Quality, WGI (bundled in V-Dem, QA only)",
    "e_wbgi_rle": "Rule of Law, WGI (bundled in V-Dem, QA only)",
    "e_wbgi_vae": "Voice & Accountability, WGI (bundled in V-Dem, QA only)",
    "e_wbgi_pve": "Political Stability & Absence of Violence, WGI (bundled in V-Dem, QA only)",
}
# CUT in V7 (was VDEM_ECON_CONTROL_INDICATORS: e_gdppc/e_pop/e_miurbani/
# e_pematmor/e_civil_war). e_gdppc duplicated WDI's own GDP/capita already in
# this panel; the rest never fed any paper claim. See the markdown note above
# if you want to reinstate any of these for a different purpose.

VDEM_INDICATORS = {**VDEM_DEMOCRACY_INDICATORS, **VDEM_RESILIENCE_OUTCOME_INDICATORS,
                   **VDEM_CORRUPTION_INDICATORS, **VDEM_VOICE_EXIT_INDICATORS,
                   **VDEM_STATECAPACITY_INDICATORS, **VDEM_SOCIALMEDIA_INDICATORS,
                   **VDEM_WGI_CROSSCHECK_INDICATORS}

# Category 2 -- migration / remittances (World Bank WDI, classic v2 API).
MIGRATION_INDICATORS = {
    "BX.TRF.PWKR.CD.DT": "Personal Remittances Received (current US$)",
    "BX.TRF.PWKR.DT.GD.ZS": "Personal Remittances Received (% of GDP)",
    "SM.POP.NETM": "Net Migration (5-year UN DESA estimate)",
    "SM.POP.TOTL": "International Migrant Stock, Total (UN DESA)",
}
# Indicators whose raw values are large enough to need billions/millions-style
# axis formatting rather than raw scientific notation -- used by plot_trend_panel.
BIG_NUMBER_INDICATORS = {"BX.TRF.PWKR.CD.DT", "SM.POP.NETM", "SM.POP.TOTL"}

# Category 3 -- socio-economic (World Bank WDI, classic v2 API).
SOCIOECON_INDICATORS = {
    "NY.GDP.PCAP.CD": "GDP per Capita (current US$)",
    "SE.SEC.NENR": "Secondary School Enrollment, Net (%)",
    "SL.UEM.1524.ZS": "Youth Unemployment, Ages 15-24 (% of labor force, ILO modelled)",
}

WDI_START, WDI_END = 1990, 2024
INDICATOR_LABELS = {**WGI_INDICATORS, **VDEM_INDICATORS, **MIGRATION_INDICATORS, **SOCIOECON_INDICATORS}

# Short, compact display variants for space-constrained figure elements
# (heatmap tick labels, in-legend text) where the full INDICATOR_LABELS text
# is too long to lay out cleanly. Everywhere else, INDICATOR_LABELS is the
# canonical name to use in titles, axis labels, and paper text.
SHORT_LABELS = {
    "GOV_WGI_VA": "Voice & Accountability",
    "GOV_WGI_PV": "Political Stability",
    "GOV_WGI_GE": "Government Effectiveness",
    "GOV_WGI_RQ": "Regulatory Quality",
    "GOV_WGI_RL": "Rule of Law",
    "GOV_WGI_CC": "Control of Corruption",
    "v2clfmove": "Freedom of Movement (exit)",
    "v2x_regime": "Democratic Resilience",
    "v2x_corr": "Political Corruption (V-Dem)",
    "v2x_libdem": "Liberal Democracy (V-Dem)",
    "BX.TRF.PWKR.CD.DT": "Remittances (US$)",
    "BX.TRF.PWKR.DT.GD.ZS": "Remittances (% GDP)",
    "SM.POP.NETM": "Net Migration",
    "SM.POP.TOTL": "Migrant Stock",
}

def label_for(code_, short=False):
    """Human-readable name for any indicator code used in this notebook.
    Falls back to the raw code only if it truly isn't registered anywhere --
    that fallback is what V6 always did, so this makes it the exception
    rather than the default."""
    if short and code_ in SHORT_LABELS:
        return SHORT_LABELS[code_]
    return INDICATOR_LABELS.get(code_, SHORT_LABELS.get(code_, code_))

def country_name(iso3):
    return COUNTRY_NAMES.get(iso3, iso3)

print(f"{len(COUNTRIES)} countries x {len(INDICATOR_LABELS)} indicators configured "
      f"({len(VDEM_INDICATORS)} of them from V-Dem, trimmed from 33 to {len(VDEM_INDICATORS)} in V7 "
      "-- see the markdown cell above for exactly what was cut).")

In [ ]:
# 4 · Fetch functions for WDI (migration + socio-economic) -- classic v2 API,
# confirmed working. WGI is handled separately below via a manual file load.
WB_BASE = "https://api.worldbank.org/v2"

def _wb_url(countries, indicators, start_year, end_year, page=1, per_page=20000):
    c = ";".join(countries)
    i = ";".join(indicators)
    return (f"{WB_BASE}/country/{c}/indicator/{i}"
            f"?format=json&date={start_year}:{end_year}&per_page={per_page}&page={page}")

def _parse_wb_json(payload):
    if isinstance(payload, dict):  # WB returns a dict (error message) instead of a list on failure
        raise RuntimeError(payload)
    if not payload or len(payload) < 2 or payload[1] is None:
        return pd.DataFrame(columns=["country", "iso3", "indicator", "year", "value"]), {"pages": 1}
    meta, records = payload[0], payload[1]
    rows = [{
        "country": r["country"]["value"], "iso3": r["countryiso3code"],
        "indicator": r["indicator"]["id"], "year": int(r["date"]), "value": r["value"],
    } for r in records]
    return pd.DataFrame(rows), meta

def fetch_worldbank(countries, indicators, start_year, end_year,
                    pause=0.4, max_retries=4):
    sess = requests.Session()
    all_frames = []
    for indicator in indicators:
        page = 1
        total_pages = 1
        while page <= total_pages:
            url = _wb_url(countries, [indicator], start_year, end_year, page=page)
            payload = None
            for attempt in range(max_retries):
                try:
                    resp = sess.get(url, timeout=30)
                    resp.raise_for_status()
                    payload = resp.json()
                    break
                except Exception:
                    time.sleep(pause * (attempt + 1))
            if payload is None:
                raise RuntimeError(f"Failed to fetch\n{url}")
            df, meta = _parse_wb_json(payload)
            all_frames.append(df)
            total_pages = int(meta.get("pages", 1))
            page += 1
            time.sleep(pause)
    out = pd.concat(all_frames, ignore_index=True)
    out = out.dropna(subset=["value"])
    return out.sort_values(["indicator", "iso3", "year"]).reset_index(drop=True)

def long_to_wide(df_long):
    wide = df_long.pivot_table(index=["iso3", "year"], columns="indicator", values="value").reset_index()
    wide.columns.name = None
    return wide.sort_values(["iso3", "year"]).reset_index(drop=True)

print("WDI fetch functions ready (classic v2 API).")

In [ ]:
# 4b · WGI loader -- reads a manually downloaded DataBank/WGI export (CSV or
# Excel) and returns the same tidy schema fetch_worldbank() produces, so
# everything downstream needs zero changes regardless of where WGI came from.
import re

WGI_NAME_MATCH = {
    "GOV_WGI_VA": ["voice and account"],
    "GOV_WGI_PV": ["political stability"],
    "GOV_WGI_GE": ["government effectiveness"],
    "GOV_WGI_RQ": ["regulatory quality"],
    "GOV_WGI_RL": ["rule of law"],
    "GOV_WGI_CC": ["control of corruption"],
}
MISSING_MARKERS = {"..", "", "na", "n/a", "#n/a", "..."}

def _match_indicator(series_name):
    s = str(series_name).strip().lower()
    for code_, needles in WGI_NAME_MATCH.items():
        if any(n in s for n in needles):
            return code_
    return None

def load_wgi_manual(path, countries=None, verbose=True):
    if str(path).lower().endswith((".xlsx", ".xls")):
        raw = pd.read_excel(path)
    else:
        raw = pd.read_csv(path)

    if verbose:
        print(f"Raw file shape: {raw.shape}")
        print("Raw columns (first 8):", list(raw.columns)[:8], "...")

    cols_lower = {c: str(c).strip().lower() for c in raw.columns}

    def find_col(*keywords):
        for c, cl in cols_lower.items():
            if all(k in cl for k in keywords):
                return c
        return None

    series_name_col = find_col("series", "name")
    country_name_col = find_col("country", "name")
    country_code_col = find_col("country", "code")

    missing = [n for n, c in [("Series Name", series_name_col),
                               ("Country Name", country_name_col),
                               ("Country Code", country_code_col)] if c is None]
    if missing:
        raise ValueError(
            f"Couldn't find expected column(s) {missing}. Actual columns are: {list(raw.columns)}\n"
            "DataBank sometimes adds a couple of title rows above the real header -- if so, "
            "re-run with e.g. pd.read_csv(path, skiprows=N) adjusted."
        )

    year_cols = [c for c in raw.columns if re.match(r"^\d{4}", str(c).strip())]
    if verbose:
        print(f"Detected {len(year_cols)} year columns "
              f"({year_cols[0]} ... {year_cols[-1]})" if year_cols else "No year columns detected!")

    df = raw.dropna(subset=[country_code_col]).copy()
    df = df[df[country_code_col].astype(str).str.len() == 3]

    df["indicator"] = df[series_name_col].apply(_match_indicator)
    unmatched = sorted(set(df.loc[df["indicator"].isna(), series_name_col].astype(str)))
    if unmatched and verbose:
        print(f"\nNOTE: {len(unmatched)} series in the file weren't one of the 6 WGI "
              f"dimensions and were skipped: {unmatched}")
    df = df.dropna(subset=["indicator"])

    missing_dims = sorted(set(WGI_NAME_MATCH) - set(df["indicator"].unique()))
    if missing_dims and verbose:
        print(f"\nWARNING: these WGI dimensions were NOT found in the file at all: {missing_dims}")

    long_df = df.melt(id_vars=[country_name_col, country_code_col, "indicator"],
                       value_vars=year_cols, var_name="year_raw", value_name="value")
    long_df["year"] = long_df["year_raw"].str.extract(r"(\d{4})").astype(int)
    long_df["value"] = long_df["value"].astype(str).str.strip().replace(list(MISSING_MARKERS), np.nan)
    long_df["value"] = pd.to_numeric(long_df["value"], errors="coerce")
    long_df = long_df.rename(columns={country_name_col: "country", country_code_col: "iso3"})
    long_df = long_df.dropna(subset=["value"])

    if countries:
        long_df = long_df[long_df["iso3"].isin(countries)]

    out = long_df[["country", "iso3", "indicator", "year", "value"]].sort_values(
        ["indicator", "iso3", "year"]).reset_index(drop=True)

    if verbose:
        print(f"\nParsed tidy shape: {out.shape}")
        print("Rows per indicator:\n", out.groupby("indicator").size())
        print("Rows per country:\n", out.groupby("iso3").size())
        if len(out):
            print(f"Year range: {out['year'].min()}-{out['year'].max()}")
    return out

print("load_wgi_manual() ready.")

In [ ]:
# 5 · Load WGI (categories 1 & 4) from your downloaded file
# Get the file first from one of the two links in the markdown cell above,
# then run this cell -- it will prompt you to pick the file from your computer.
try:
    from google.colab import files
    print("Select the WGI file you downloaded (CSV or .xlsx):")
    uploaded = files.upload()
    wgi_manual_path = list(uploaded.keys())[0]
except ImportError:
    wgi_manual_path = "wgi_export.csv"
    print(f"Not running in Colab -- looking for '{wgi_manual_path}' in the working directory. "
          "Edit this path if your file is named differently.")

wgi_long = load_wgi_manual(wgi_manual_path, countries=COUNTRIES)
wgi_long.head()

In [ ]:
# 5b · V-Dem loader function -- reads ONLY the columns we need from your full
# V-Dem country-year file (it has 4,000+ columns; this never loads them all
# into memory), filters to our 6 countries + year window, and returns the
# SAME tidy schema as fetch_worldbank()/load_wgi_manual() above.
def load_vdem_manual(path, indicator_codes, countries=None,
                      start_year=None, end_year=None, verbose=True):
    id_cols = ["country_name", "country_text_id", "year"]
    wanted = list(dict.fromkeys(id_cols + list(indicator_codes)))

    header = pd.read_csv(path, nrows=0).columns.tolist()
    present = [c for c in wanted if c in header]
    missing = [c for c in wanted if c not in header]
    if missing and verbose:
        print(f"NOTE: {len(missing)} requested V-Dem column(s) not found in this file "
              f"and will be skipped (check spelling / your V-Dem version): {missing}")

    raw = pd.read_csv(path, usecols=present, low_memory=False)
    if verbose:
        print(f"Read {raw.shape[0]:,} rows x {raw.shape[1]} columns from the full V-Dem "
              f"file (only the {len(present)} columns actually requested).")

    df = raw.copy()
    if countries:
        not_found = sorted(set(countries) - set(df["country_text_id"].unique()))
        if not_found and verbose:
            print(f"WARNING: these requested countries were NOT found by country_text_id: {not_found}")
        df = df[df["country_text_id"].isin(countries)]

    if start_year is not None:
        df = df[df["year"] >= start_year]
    if end_year is not None:
        df = df[df["year"] <= end_year]

    value_cols = [c for c in present if c not in id_cols]
    long_df = df.melt(id_vars=id_cols, value_vars=value_cols,
                       var_name="indicator", value_name="value")
    long_df = long_df.rename(columns={"country_name": "country", "country_text_id": "iso3"})
    long_df["value"] = pd.to_numeric(long_df["value"], errors="coerce")
    long_df = long_df.dropna(subset=["value"])

    out = long_df[["country", "iso3", "indicator", "year", "value"]].sort_values(
        ["indicator", "iso3", "year"]).reset_index(drop=True)

    if verbose:
        print(f"\nParsed tidy shape: {out.shape}")
        print("Rows per indicator:\n", out.groupby("indicator").size())
        print("Rows per country:\n", out.groupby("iso3").size())
        if len(out):
            print(f"Year range: {out['year'].min()}-{out['year'].max()}")
    return out

print("load_vdem_manual() ready.")

In [ ]:
# 5c · Load V-Dem (enrichment -- robustness-check + deeper sub-components
# alongside WGI, NOT a trust/migration-intention substitute -- see Section 0b)
#
# Get the "Country-Year: V-Dem Full+Others" CSV from
# https://v-dem.net/data/the-v-dem-dataset/ (free, no login/registration).
try:
    from google.colab import files
    print("Select your V-Dem Country-Year CSV (the full file -- this cell only reads "
          f"the {len(VDEM_INDICATORS)} columns it needs from it):")
    uploaded = files.upload()
    vdem_path = list(uploaded.keys())[0]
except ImportError:
    vdem_path = "V-Dem-CY-Full+Others.csv"
    print(f"Not running in Colab -- looking for '{vdem_path}' in the working directory. "
          "Edit this path if your file is named differently.")

vdem_long = load_vdem_manual(vdem_path, list(VDEM_INDICATORS.keys()),
                              countries=COUNTRIES, start_year=WDI_START, end_year=WDI_END)
vdem_long.head()

In [ ]:
# 6 · Pull migration + socio-economic indicators (categories 2 & 3)
migration_long = fetch_worldbank(COUNTRIES, list(MIGRATION_INDICATORS.keys()), WDI_START, WDI_END)
socioecon_long = fetch_worldbank(COUNTRIES, list(SOCIOECON_INDICATORS.keys()), WDI_START, WDI_END)
print("Migration rows:", len(migration_long), "| Socioeconomic rows:", len(socioecon_long))
migration_long.head()

In [ ]:
# 7 · Merge into one master country-year panel + missingness check
long_all = pd.concat([wgi_long, vdem_long, migration_long, socioecon_long], ignore_index=True)
panel = long_to_wide(long_all)
panel["country"] = panel["iso3"].map(COUNTRY_NAMES)

print("Master panel shape:", panel.shape)
missingness = (panel.set_index(["country", "year"])
                     .drop(columns=["iso3"])
                     .isna().groupby(level=0).mean().round(2) * 100)
print("\n% missing by country x indicator (0 = fully observed):")
display(missingness)

# SM.POP.TOTL (migrant stock) is a 5-year UN DESA series -- most years are
# genuinely missing, not a data-quality problem.
print("\nSM.POP.TOTL non-missing years, all countries:",
      sorted(panel.loc[panel["SM.POP.TOTL"].notna(), "year"].unique()))
print("Rows per indicator (sanity check):\n", long_all.groupby("indicator").size())

In [ ]:
# 7b · INTERNAL QA -- not a paper figure or result. Manual WGI download vs.
# V-Dem's bundled WGI estimates: V-Dem's "Full+Others" release folds in the
# same World Bank WGI "Estimate" series as external variables (e_wbgi_*), so
# these two SHOULD closely agree almost everywhere they both have data. A LOW
# correlation here would flag a parsing problem in load_wgi_manual() (Section
# 4b) or a country/year mismatch -- not a real disagreement between two
# governance datasets. Nothing below this cell should cite this output.
wgi_vs_vdem_map = {
    "GOV_WGI_GE": "e_wbgi_gee", "GOV_WGI_CC": "e_wbgi_cce", "GOV_WGI_RQ": "e_wbgi_rqe",
    "GOV_WGI_RL": "e_wbgi_rle", "GOV_WGI_VA": "e_wbgi_vae", "GOV_WGI_PV": "e_wbgi_pve",
}
print("Internal QA -- manual WGI file vs. V-Dem-bundled WGI, Pearson r over overlapping country-years:")
for wgi_code, vdem_code in wgi_vs_vdem_map.items():
    if wgi_code in panel.columns and vdem_code in panel.columns:
        both = panel[[wgi_code, vdem_code]].dropna()
        if len(both) >= 5:
            r = both[wgi_code].corr(both[vdem_code])
            flag = "" if r > 0.9 else "  <-- check this one, expected > ~0.9"
            print(f"  {label_for(wgi_code, short=True):26s} vs {vdem_code:12s}: r={r:.3f} over {len(both)} country-years{flag}")
        else:
            print(f"  {label_for(wgi_code, short=True):26s} vs {vdem_code:12s}: not enough overlapping rows to check yet")
    else:
        print(f"  {label_for(wgi_code, short=True):26s} vs {vdem_code:12s}: one or both columns not in panel yet "
              "-- run Sections 5c and 7 first")

In [ ]:
# 8 · Save the master panel
# (V7: removed the direct UN DESA cross-check that lived here in V6 -- it
# fetched the raw workbook and printed a preview but never resolved into an
# actual comparison number, so it was an unfinished step rather than a
# distinct finding. SM.POP.TOTL is already UN-DESA-sourced via the WDI
# mirror above, and Section 7b already gives a real, finished cross-check
# for the governance side.)
panel_path = f"{DATA_DIR}/south_asia_panel.csv"
panel.to_csv(panel_path, index=False)
print("Saved ->", panel_path)

# SM.POP.TOTL_filled: a linearly-interpolated version of the 5-year migrant-
# stock series, for TREND PLOTTING ONLY (Figure B, Section 13). Interior gaps
# (between two known 5-year checkpoints) are filled with a straight line
# between them -- a more honest depiction of a gradually-changing stock than
# a flat step function -- while the leading/trailing edges (before the first
# or after the last checkpoint, where there's no second point to interpolate
# toward) hold the nearest known value flat, the standard convention when
# there's nothing newer to go on.
#
# BUG FIX: this column already existed, but Figure B's plotting cell (Section
# 13) was never actually updated to use it -- it still plotted the raw,
# genuinely-sparse SM.POP.TOTL directly (only ~7 real points across 35 years),
# so the migrant-stock panel rendered as completely blank: every real data
# point sat between two NaNs with no marker set, so matplotlib had no
# adjacent point to draw a line segment to or from. Confirmed by reproducing
# the exact pattern on synthetic 5-year-checkpoint data before writing this fix.
#
# Stored as a SEPARATE column (never overwriting the real SM.POP.TOTL)
# precisely so it can't accidentally end up in a correlation or Granger cell,
# where the interpolation would fabricate autocorrelation that isn't in the
# real data. Use SM.POP.TOTL (raw), never SM.POP.TOTL_filled, in any
# corr/Granger cell -- Sections 16b/16d already correctly do this.
panel["SM.POP.TOTL_filled"] = (
    panel.groupby("iso3")["SM.POP.TOTL"]
         .transform(lambda s: s.interpolate(method="linear", limit_direction="both"))
)
print("Added SM.POP.TOTL_filled (plotting only) -- now actually wired into Figure B below.")

## 9 · Enrichment data sources: what to use and how to access it

The trust/perception side of the theory needed a real citizen-level survey -- that
conclusion hasn't changed -- and Section 21 below now uses exactly that (real ABS
Wave 1/2 microdata). Here's the full picture.

| Source | Gives you | Access | Level |
|---|---|---|---|
| **Asian Barometer Survey (ABS)** | Trust, corruption perception, satisfaction with democracy, economic evaluations | Free, data-request form: `https://www.asianbarometer.org` → Data → Merged Data Request. Covers Nepal, India, Pakistan, Bangladesh, Sri Lanka (5 of our 6 -- no Bhutan). **Status: real Wave 1 (2005) / Wave 2 (2013) data loaded and used below (Section 21).** |
| **Asia Foundation "Survey of the Nepali People"** | Trust by federal/provincial/local government, by province -- Nepal-only, denser | Microdata for 2017/2018/2020: `https://dataverse.ada.edu.au/dataset.xhtml?persistentId=doi:10.26193/K9QPBH` |
| **V-Dem** | Expert-coded democracy/corruption/civil-liberties indices | **Loaded -- Section 5b/5c.** Free bulk download, `v-dem.net`. All 6 countries, including Bhutan. |
| **World Values Survey** | Same category as ABS | Skip -- Nepal isn't consistently fielded across WVS waves. |
| **Global Data Lab Subnational HDI (SHDI)** | District-level HDI/education/income, 1990-2022, all 6 countries | Free registration (~5 min): `https://globaldatalab.org/shdi/download/`. **Not loaded in this notebook** -- this is the source that would be needed for RQ2 (see Section 0b); out of scope here by design, not by oversight. |
| **Nepal DoFE district labor-permit series** | District-wise realized labor migration, Nepal only | `dofe.gov.np` annual reports -- typically PDF/table extraction. **Not loaded** -- same RQ2/brain-circulation scope boundary as above. |

**The harmonization problem is real and worth taking seriously before merging anything:**
ABS uses its own province/region codes, SHDI uses `GDLCODE` district identifiers, and
DoFE uses pre-2015 district names. Nepal's 2015 federal restructuring means none of
these line up automatically -- budget this crosswalk as its own task if you extend
this notebook toward RQ2 later.

## 10 · Descriptive trends — paper-sync

**Where this goes:** Section 2 (a sourced, reproducible replacement/supplement for
Fig. 1's remittance and governance claims) and Section 6 (comparative backdrop for
Table 3/Table 4).

Nepal is drawn in bold; the five comparators sit underneath in their own distinct
colors (not just "Nepal vs. grey") so any single figure is readable on its own.
Countries and indicators are labeled with their real names throughout — see
`label_for()` and the legend fix below.

In [ ]:
# 11 · Plotting setup (styled for direct paper inclusion: 300dpi PNG + vector PDF)
COUNTRY_COLORS = {
    "NPL": "#B23A48",  # Nepal -- kept as the accent color, still visually emphasized
    "IND": "#2A6F77",
    "BGD": "#C98A2C",
    "PAK": "#5B6EE1",
    "LKA": "#4F9D69",
    "BTN": "#8B5FBF",
}
GRID = "#E3E6E8"
plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 300, "font.size": 10.5,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "axes.edgecolor": "#333333",
})

def _big_number_formatter(x, pos=None):
    """V7: renders large USD/count axis values as e.g. '140B' / '7.2M' instead
    of raw scientific notation (1.4e11) -- one of the 'proper scales' fixes."""
    ax = abs(x)
    if ax >= 1e9:
        return f"{x/1e9:.1f}B" if (ax / 1e9) < 10 else f"{x/1e9:.0f}B"
    if ax >= 1e6:
        return f"{x/1e6:.1f}M" if (ax / 1e6) < 10 else f"{x/1e6:.0f}M"
    if ax >= 1e3:
        return f"{x/1e3:.0f}K"
    return f"{x:.0f}"

def plot_trend_panel(wide_df, indicators, focal_iso3=FOCAL, title=None, ncols=3, savepath=None):
    """indicators: list of (code, label) or (code, label, format_hint) tuples.
    format_hint="big_number" applies the billions/millions axis formatter.
    V7 fix: legend now shows real country names (via COUNTRY_NAMES), not ISO3
    codes -- previously the only place a raw code leaked into a figure's
    legend rather than its title."""
    n = len(indicators); ncols = min(ncols, n); nrows = -(-n // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.6 * ncols, 3.4 * nrows), squeeze=False)
    axes_flat = axes.flatten()
    focal_name = country_name(focal_iso3)
    for ax, item in zip(axes_flat, indicators):
        col, label = item[0], item[1]
        fmt_hint = item[2] if len(item) > 2 else None
        for iso3, grp in wide_df.groupby("iso3"):
            grp = grp.sort_values("year")
            is_focal = iso3 == focal_iso3
            ax.plot(grp["year"], grp[col], label=country_name(iso3),
                     color=COUNTRY_COLORS.get(iso3, "#999999"),
                     linewidth=2.4 if is_focal else 1.4,
                     alpha=1.0 if is_focal else 0.85,
                     zorder=5 if is_focal else 3)
        ax.set_title(label, fontsize=10.5, loc="left")
        ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True, nbins=6))
        if fmt_hint == "big_number":
            ax.yaxis.set_major_formatter(mticker.FuncFormatter(_big_number_formatter))
    for ax in axes_flat[n:]:
        ax.axis("off")
    handles, labels_ = axes_flat[0].get_legend_handles_labels()
    by_label = dict(zip(labels_, handles))
    ordered = sorted(by_label, key=lambda name: (name != focal_name, name))  # focal country listed first
    fig.legend([by_label[c] for c in ordered], ordered, loc="lower center",
               ncol=min(len(by_label), 6), frameon=False, bbox_to_anchor=(0.5, -0.03))
    if title:
        fig.suptitle(title, fontsize=13, y=1.02, fontweight="bold")
    fig.tight_layout()
    if savepath:
        fig.savefig(savepath, bbox_inches="tight")
        fig.savefig(savepath.rsplit(".", 1)[0] + ".pdf", bbox_inches="tight")
    return fig

print("Plot helper ready -- one color per country, legend uses real country names:",
      {k: country_name(k) for k in COUNTRY_COLORS})

### Figure A — Worldwide Governance Indicators — paper-sync

**Where this goes:** Section 2 (Fig. 1 replacement/supplement) and Section 5.1/5.2
(the "implementation gap" argument).

**Suggested paper text:**
> Figure [X] traces the six World Bank Worldwide Governance Indicators for Nepal
> against five South Asian comparators, 1996–2024. Nepal's Voice & Accountability
> and Political Stability & Absence of Violence show a sustained improvement
> following the post-2006 transition described in Section 5.1, while Government
> Effectiveness, Regulatory Quality, and Control of Corruption remain comparatively
> flat or volatile over the same period — consistent with the "implementation gap"
> documented qualitatively in Sections 5.1–5.2.

**What it infers / why it supports the paper:** separating the six WGI dimensions
shows *which* parts of governance moved with the democratic transition (the
political/civil ones) and which didn't (the administrative/regulatory ones) — this
is a quantitative anchor for the paper's specific claim that formal democratic
architecture outpaced service-delivery competence, not a generic "governance is bad"
statement.

In [ ]:
# 12 · Figure A -- WGI governance/trust trends (categories 1 & 4)
gov_cols = [(c, l) for c, l in WGI_INDICATORS.items()]
fig_a = plot_trend_panel(panel, gov_cols,
                          title="Worldwide Governance Indicators, Nepal vs. South Asia (1996-2024)",
                          savepath=f"{FIG_DIR}/fig_a_governance_trends.png")
plt.show()

### Figure A2 + Democratic Resilience — paper-sync

**Where this goes:** Figure A2 (exit/civic-space trends) → Section 3.2 (Exit-Voice-
Loyalty) and Section 5.6/5.7. The new `v2x_regime` figure → Section 5.7/Table 1
groundwork, and directly labels **Fig. 2's own terminal node, "Democratic
Resilience."**

**Suggested paper text:**
> Figure [X] traces V-Dem's liberal democracy, political corruption, freedom of
> foreign movement ("exit"), and core civil society indices for Nepal against its
> comparators. Figure [X+1] plots V-Dem's Regimes-of-the-World classification —
> this notebook's direct operationalization of "democratic resilience" — showing
> [describe whether/when Nepal's regime category shifts once you have real output].

**What it infers / why it supports the paper:** V6 loaded `v2x_regime` — the
paper's own stated stand-in for democratic resilience — but never gave it a figure
of its own; it sat as one unlabeled line inside a four-panel chart. Since the
paper's entire conceptual model (Fig. 2) ends at this construct, it earns a
dedicated trend rather than a cameo.

In [ ]:
# 12b · Figure A2 -- V-Dem governance & civic-space trends
vdem_plot_cols = [
    ("v2x_libdem", "Liberal Democracy Index"),
    ("v2x_corr", "Political Corruption Index"),
    ("v2clfmove", "Freedom of Foreign Movement (exit option)"),
    ("v2xcs_ccsi", "Core Civil Society Index"),
]
vdem_plot_cols = [(c, l) for c, l in vdem_plot_cols if c in panel.columns]
if vdem_plot_cols:
    fig_a2 = plot_trend_panel(panel, vdem_plot_cols, ncols=2,
                               title="V-Dem Governance & Civic Space, Nepal vs. South Asia",
                               savepath=f"{FIG_DIR}/fig_a2_vdem_trends.png")
    plt.show()
else:
    print("No V-Dem columns in panel yet -- run Sections 5b/5c/7 first.")

In [ ]:
# 12c · NEW in V7 -- Figure G: Democratic Resilience (V-Dem Regimes of the World)
# Gets its own figure because it is Fig. 2's own terminal node, not because it
# needs more visual real estate than anything else here.
REGIME_CATEGORY_LABELS = {0: "Closed\nautocracy", 1: "Electoral\nautocracy",
                           2: "Electoral\ndemocracy", 3: "Liberal\ndemocracy"}

if "v2x_regime" in panel.columns:
    fig_g, ax = plt.subplots(figsize=(8, 4.6))
    for iso3, grp in panel.groupby("iso3"):
        grp = grp.sort_values("year").dropna(subset=["v2x_regime"])
        if grp.empty:
            continue
        is_focal = iso3 == FOCAL
        ax.plot(grp["year"], grp["v2x_regime"], label=country_name(iso3),
                color=COUNTRY_COLORS.get(iso3, "#999999"),
                linewidth=2.6 if is_focal else 1.5, alpha=1.0 if is_focal else 0.85,
                marker="o", markersize=3.2, zorder=5 if is_focal else 3)
    ax.set_yticks(list(REGIME_CATEGORY_LABELS.keys()))
    ax.set_yticklabels(list(REGIME_CATEGORY_LABELS.values()), fontsize=9)
    ax.set_ylim(-0.4, 3.4)
    ax.set_ylabel("Regime category (V-Dem Regimes of the World)")
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True, nbins=8))
    ax.set_title("Democratic Resilience — Regimes of the World classification (V-Dem)",
                 fontsize=12, loc="left", fontweight="bold")
    handles, labels_ = ax.get_legend_handles_labels()
    by_label = dict(zip(labels_, handles))
    ordered = sorted(by_label, key=lambda name: (name != country_name(FOCAL), name))
    ax.legend([by_label[c] for c in ordered], ordered, frameon=False, fontsize=8.5,
              loc="upper left", ncol=2, bbox_to_anchor=(0.0, -0.16))
    fig_g.tight_layout()
    fig_g.savefig(f"{FIG_DIR}/fig_g_democratic_resilience_regime.png", bbox_inches="tight")
    fig_g.savefig(f"{FIG_DIR}/fig_g_democratic_resilience_regime.pdf", bbox_inches="tight")
    plt.show()
else:
    print("v2x_regime not in panel yet -- run Sections 5b/5c/7 first.")

### Figure B — Migration & Remittances — paper-sync

**Where this goes:** Section 2 (Fig. 1 remittance-trend replacement) and Section
5.3/5.4 (exit *and* loyalty, simultaneously).

**Suggested paper text:**
> Figure [X] traces remittance and migration-stock trends for Nepal against
> comparators, 1990–2024. Nepal's remittances rise from under 5% to over 25% of
> GDP over this period even as governance indicators (Figure A) remain comparatively
> flat — a pattern consistent with Section 5.4's argument that remittance inflows
> mask, rather than resolve, structural governance and labor-market weaknesses.

**What it infers — the "loyalty" reading, not just "exit":** Section 3.2 argues
exit and loyalty are simultaneous responses to the same legitimacy gap, not
substitutes. The same rising-remittances line that indexes *exit* (realized
emigration) is also direct evidence of *loyalty* in Hirschman's specific sense —
continued financial engagement with the home country despite dissatisfaction. Cite
this figure for both claims; it is genuinely evidence for each.

**Caveat for the migrant-stock panel specifically:** UN DESA only publishes this
series every 5 years (~7 real data points across the whole period), so the line
shown is linearly interpolated between real checkpoints for visual continuity —
disclosed directly in that panel's own title. Describe it in the paper as "5-year
UN DESA estimates, interpolated for display," not as annual-resolution data, and
don't read short-run wiggles in that specific line as real year-to-year movement.

In [ ]:
# 13 · Figure B -- migration / remittance trends (category 2)
# BUG FIX: SM.POP.TOTL is a genuine 5-year-only UN DESA series (~7 real points
# across 35 years) -- plotted directly, every point is isolated between two
# NaNs and nothing renders (confirmed: this is why the migrant-stock panel came
# back completely blank). Plot SM.POP.TOTL_filled (Section 8) here instead --
# the label is unchanged, but a one-line disclosure is appended to THIS panel's
# title only, so the figure itself discloses the interpolation without
# changing the canonical indicator name used anywhere else in the notebook.
mig_cols = []
for c, l in MIGRATION_INDICATORS.items():
    fmt_hint = "big_number" if c in BIG_NUMBER_INDICATORS else None
    if c == "SM.POP.TOTL":
        mig_cols.append(("SM.POP.TOTL_filled", l + " (interpolated between 5-yr checkpoints)", fmt_hint))
    else:
        mig_cols.append((c, l, fmt_hint))

fig_b = plot_trend_panel(panel, mig_cols, ncols=2,
                          title="Migration & Remittance Trends, Nepal vs. South Asia",
                          savepath=f"{FIG_DIR}/fig_b_migration_trends.png")
plt.show()

### Figure C — Socioeconomic Trends — paper-sync

**Where this goes:** Section 2 backdrop and Section 5.2/5.3 (education-to-work gap,
youth unemployment).

**Suggested paper text:**
> Figure [X] situates Nepal's GDP per capita, secondary enrollment, and youth
> unemployment against the same comparators. [Fill in the specific pattern once run
> — e.g., whether Nepal's youth-unemployment line tracks or diverges from
> comparators over the SEE-reform period discussed in Section 5.2.]

**What it infers:** provides the socio-economic leg of the "objective conditions"
backdrop that Definition 1.1 says gets filtered through citizens' perception before
it becomes a migration/trust outcome — i.e., this is the input side, and Section 21
(ABS) is the perception side.

In [ ]:
# 14 · Figure C -- socio-economic trends (category 3)
socio_cols = [(c, l) for c, l in SOCIOECON_INDICATORS.items()]
fig_c = plot_trend_panel(panel, socio_cols,
                          title="Socio-economic Trends, Nepal vs. South Asia",
                          savepath=f"{FIG_DIR}/fig_c_socioeconomic_trends.png")
plt.show()

## 15 · Co-movement — retargeted in V7 — paper-sync

**Where this goes:** Section 3.2 (Exit-Voice-Loyalty) and Section 5.3 (migration as
anticipatory response to perceived opportunity).

**What changed and why:** V6 tested *Control of Corruption* against *remittances*
here — a real, defensible pairing for Section 5.1's corruption narrative, but not
the specific relationship the paper's own theory names. Fig. 2 and Section 3.2
identify **`v2clfmove` (freedom of foreign movement) as the literal "exit" variable**
in Hirschman's framework — it was loaded in V6 but never tested against anything.
V7 makes it the **primary** pairing (against net migration / migrant stock, the
closest available "realized exit" outcomes) and keeps corruption-vs-remittances as a
**secondary** check, since corruption is still directly relevant to Section 5.1.

**Suggested paper text (once run):**
> A stationarity-gated Granger test [does/does not] find that formal freedom of
> foreign movement statistically precedes realized migration outcomes for Nepal
> (1996–2024); [a parallel check for Control of Corruption against remittances
> finds/does not find...]. Both are descriptive, country-level precedence checks,
> not a test of individual migration decision-making (Definition 1.1).

Two methodological upgrades carried over from V6, unchanged: (1) **stationarity
first** (ADF+KPSS gate, since several of these series trend for 25+ years and a
naive Granger test on two trending series can manufacture false "precedence" — the
self-check below validates this on synthetic independent random walks before
trusting it on real data); (2) **panel, not just Nepal** (Section 16, a pooled
6-country Dumitrescu–Hurlin-style test with a permutation p-value, more
trustworthy than the asymptotic version at N=6).

In [ ]:
# 16a · Stationarity testing + a stationarity-gated Granger function
from statsmodels.tsa.stattools import adfuller, kpss

def stationarity_verdict(series, regression="ct"):
    s = series.dropna()
    _, adf_p, *_ = adfuller(s, autolag="AIC", regression=regression)
    try:
        _, kpss_p, *_ = kpss(s, regression=regression, nlags="auto")
    except Exception:
        kpss_p = np.nan
    adf_says_stationary = adf_p < 0.05
    kpss_says_stationary = (kpss_p > 0.05) if not np.isnan(kpss_p) else None
    if adf_says_stationary and kpss_says_stationary:
        verdict = "stationary"
    else:
        verdict = "non-stationary"
    return {"adf_p": round(adf_p, 4), "kpss_p": round(kpss_p, 4) if not np.isnan(kpss_p) else None,
            "verdict": verdict}

def safe_granger(x, y, maxlag=2, verbose=True):
    """Stationarity-gated Granger: differences first if either series is non-stationary."""
    df = pd.concat([y.rename("y"), x.rename("x")], axis=1).dropna()
    vx, vy = stationarity_verdict(df["x"]), stationarity_verdict(df["y"])
    used = "levels"
    if vx["verdict"] == "non-stationary" or vy["verdict"] == "non-stationary":
        df = df.diff().dropna()
        used = "first-differenced"
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        res = grangercausalitytests(df[["y", "x"]], maxlag=maxlag, verbose=False)
    pvals = {lag: round(res[lag][0]["ssr_ftest"][1], 4) for lag in res}
    if verbose:
        print(f"  x stationarity: {vx['verdict']} (ADF p={vx['adf_p']}, KPSS p={vx['kpss_p']})")
        print(f"  y stationarity: {vy['verdict']} (ADF p={vy['adf_p']}, KPSS p={vy['kpss_p']})")
        print(f"  series used for the Granger test: {used}")
    return pvals, used

# --- validate on synthetic data with KNOWN properties before trusting it on real data ---
print("=== Self-check: does the spurious-regression gate actually help? ===")
naive_sig = fixed_sig = 0
for trial in range(30):
    r = np.random.default_rng(1000 + trial)
    x_ind = pd.Series(np.cumsum(r.normal(0.1, 0.3, 27)), index=range(1996, 2023))
    y_ind = pd.Series(np.cumsum(r.normal(0.1, 0.3, 27)), index=range(1996, 2023))  # truly independent
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        naive_p = min(grangercausalitytests(pd.concat([y_ind.rename("y"), x_ind.rename("x")], axis=1),
                                             maxlag=2, verbose=False)[lag][0]["ssr_ftest"][1] for lag in [1, 2])
    naive_sig += naive_p < 0.05
    fixed_p, _ = safe_granger(x_ind, y_ind, maxlag=2, verbose=False)
    fixed_sig += min(fixed_p.values()) < 0.05
print(f"Two INDEPENDENT random walks, 30 trials: naive levels-Granger falsely significant "
      f"{naive_sig}/30 times; stationarity-gated {fixed_sig}/30 times (nominal should be ~1-2/30).")
print("(This is exactly the spurious-trend risk the report flagged -- confirmed and fixed.)\n")

In [ ]:
# 16b · Apply to the real Nepal series -- V7: exit (v2clfmove) is now the
# PRIMARY pairing; Control of Corruption vs. remittances is kept as a
# secondary check (still relevant to Section 5.1's corruption narrative).
nepal = panel[panel.iso3 == FOCAL].set_index("year").sort_index()

# ---- Figure D: correlation heatmap -- V7 fix: tick labels now use real
# indicator names (label_for, short=True) instead of raw codes like
# "GOV_WGI_GE" -- and v2clfmove is added to the matrix. ----
corr_cols = ["GOV_WGI_VA", "GOV_WGI_CC", "v2clfmove",
             "BX.TRF.PWKR.DT.GD.ZS", "SM.POP.NETM"]
corr_cols = [c for c in corr_cols if c in nepal.columns]
corr = nepal[corr_cols].corr()
readable = [label_for(c, short=True) for c in corr_cols]
corr_display = corr.copy()
corr_display.index = readable
corr_display.columns = readable

fig_d, ax = plt.subplots(figsize=(6.8, 5.6))
im = ax.imshow(corr_display.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr_display.columns))); ax.set_xticklabels(corr_display.columns, rotation=40, ha="right", fontsize=9)
ax.set_yticks(range(len(corr_display.index))); ax.set_yticklabels(corr_display.index, fontsize=9)
for i in range(corr_display.shape[0]):
    for j in range(corr_display.shape[1]):
        v = corr_display.values[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                color="white" if abs(v) > 0.55 else "black", fontsize=8.5)
fig_d.colorbar(im, ax=ax, shrink=0.8, label="Pearson r (Nepal, country-year)")
ax.set_title("Nepal: exit, governance & migration-proxy co-movement", fontsize=11.5, fontweight="bold")
fig_d.tight_layout()
fig_d.savefig(f"{FIG_DIR}/fig_d_correlation_heatmap.png", bbox_inches="tight")
fig_d.savefig(f"{FIG_DIR}/fig_d_correlation_heatmap.pdf", bbox_inches="tight")
plt.show()

# ---- PRIMARY: does formal exit-freedom move with realized migration outcomes? ----
print("\nPRIMARY -- stationarity-gated Granger, Nepal: Freedom of Foreign Movement (exit) <-> Net Migration")
g_exit = nepal[["v2clfmove", "SM.POP.NETM"]].dropna() if "v2clfmove" in nepal.columns else pd.DataFrame()
if len(g_exit) >= 8:
    print("\nfreedom of foreign movement -> net migration:")
    p_exit_xy, _ = safe_granger(g_exit["v2clfmove"], g_exit["SM.POP.NETM"], maxlag=2)
    print("  p-values by lag:", p_exit_xy)
    print("\nnet migration -> freedom of foreign movement:")
    p_exit_yx, _ = safe_granger(g_exit["SM.POP.NETM"], g_exit["v2clfmove"], maxlag=2)
    print("  p-values by lag:", p_exit_yx)
else:
    print("Not enough overlapping non-missing years (or v2clfmove not in panel) -- skipped.")

# ---- SECONDARY (kept from V6): corruption vs. remittances ----
print("\nSECONDARY -- stationarity-gated Granger, Nepal: Control of Corruption <-> Remittances (% GDP)")
g_df = nepal[["GOV_WGI_CC", "BX.TRF.PWKR.DT.GD.ZS"]].dropna()
if len(g_df) >= 8:
    print("\ncorruption -> remittances:")
    p_xy, _ = safe_granger(g_df["GOV_WGI_CC"], g_df["BX.TRF.PWKR.DT.GD.ZS"], maxlag=2)
    print("  p-values by lag:", p_xy)
    print("\nremittances -> corruption:")
    p_yx, _ = safe_granger(g_df["BX.TRF.PWKR.DT.GD.ZS"], g_df["GOV_WGI_CC"], maxlag=2)
    print("  p-values by lag:", p_yx)
else:
    print("Not enough overlapping non-missing years to run this reliably; skipped.")

print("\nReminder: both are statistical-precedence checks between national aggregates, not a "
      "test of the individual-level perception-comprehension-projection mechanism in "
      "Definition 1.1 / Fig. 2.")

### 16c · Panel Granger (Dumitrescu–Hurlin-style), pooled across all 6 countries

Real statistical power instead of 6 separate underpowered series: run the
stationarity-gated Granger test in every country, pool the Wald statistics, and test
significance both the classical (asymptotic) way and with an empirical permutation
p-value (the one to trust at N=6). **V7: exit (`v2clfmove` → net migration) is now
the primary application; corruption → remittances is kept as a secondary,
comparative check** — same rationale as Section 16b.

In [ ]:
# 16d · Panel Granger implementation + self-check + application
from scipy.stats import norm

def _wald_stat(df, lag):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        res = grangercausalitytests(df[["y", "x"]], maxlag=lag, verbose=False)
    F, pval, df_denom, df_num = res[lag][0]["ssr_ftest"]
    return df_num * F

def _prep_pair(x, y):
    df = pd.concat([y.rename("y"), x.rename("x")], axis=1).dropna()
    vx, vy = stationarity_verdict(df["x"])["verdict"], stationarity_verdict(df["y"])["verdict"]
    if vx == "non-stationary" or vy == "non-stationary":
        df = df.diff().dropna()
    return df

def panel_granger(country_series_pairs, lag=1, n_boot=2000, seed=0, shift_min_frac=0.2, verbose=True):
    """Dumitrescu-Hurlin-style pooled panel Granger test (x -> y), with a
    circular-shift-permutation empirical p-value alongside the classical
    asymptotic Z-bar. country_series_pairs: {country: (x_series, y_series)}."""
    rng = np.random.default_rng(seed)
    prepped = {c: _prep_pair(x, y) for c, (x, y) in country_series_pairs.items()
               if len(_prep_pair(x, y)) >= 2 * lag + 4}
    if len(prepped) < 3:
        if verbose:
            print(f"  Only {len(prepped)} countries had enough overlapping data -- skipping.")
        return None
    obs_stats = {c: _wald_stat(df, lag) for c, df in prepped.items()}
    W_bar_obs = np.mean(list(obs_stats.values()))
    N = len(prepped)
    Z_bar = np.sqrt(N / (2 * lag)) * (W_bar_obs - lag)
    z_bar_p = 2 * (1 - norm.cdf(abs(Z_bar)))

    boot_wbars = np.zeros(n_boot)
    for b in range(n_boot):
        stats_b = []
        for c, df in prepped.items():
            n = len(df)
            min_shift = max(1, int(shift_min_frac * n))
            hi = max(min_shift + 1, n - min_shift)
            shift = rng.integers(min_shift, hi)
            x_shifted = np.roll(df["x"].values, shift)
            try:
                stats_b.append(_wald_stat(pd.DataFrame({"y": df["y"].values, "x": x_shifted}), lag))
            except Exception:
                stats_b.append(np.nan)
        boot_wbars[b] = np.nanmean(stats_b)
    emp_p = float(np.mean(boot_wbars >= W_bar_obs))
    return {"n_countries": N, "per_country_wald": {country_name(c): round(v, 3) for c, v in obs_stats.items()},
            "W_bar": round(W_bar_obs, 4), "Z_bar_asymptotic": round(Z_bar, 4),
            "z_bar_p_asymptotic": round(z_bar_p, 4), "empirical_p_bootstrap": round(emp_p, 4)}

# --- self-check: does it correctly find nothing when there's nothing, and find a
#     real signal when all 6 countries genuinely share one? ---
print("=== Self-check on synthetic panels (known ground truth) ===")
null_pairs = {}
for i, c in enumerate(COUNTRIES):
    r = np.random.default_rng(3000 + i)
    x = pd.Series(np.cumsum(r.normal(0.1, 0.3, 27)), index=range(1996, 2023))
    y = pd.Series(np.cumsum(r.normal(0.1, 0.3, 27)), index=range(1996, 2023))
    null_pairs[c] = (x, y)
print("No true relationship in any country:", panel_granger(null_pairs, lag=1, seed=1, verbose=False))

causal_pairs = {}
for i, c in enumerate(COUNTRIES):
    r = np.random.default_rng(4000 + i)
    xv = r.normal(0, 1, 27)
    for t in range(1, 27):
        xv[t] = 0.4 * xv[t - 1] + r.normal(0, 0.8)
    yv = np.zeros(27)
    for t in range(1, 27):
        yv[t] = 0.3 * yv[t - 1] + 0.7 * xv[t - 1] + r.normal(0, 0.5)
    causal_pairs[c] = (pd.Series(xv, index=range(1996, 2023)), pd.Series(yv, index=range(1996, 2023)))
print("True x->y relationship in all 6 countries:", panel_granger(causal_pairs, lag=1, seed=2, verbose=False))
print("(first should show large p-values, second should show near-zero)\n")

# --- PRIMARY application: does exit-freedom lead net migration, pooled? ---
print("=== PRIMARY -- Freedom of Foreign Movement (exit) -> Net Migration, pooled across countries ===")
real_pairs_exit = {}
for c in COUNTRIES:
    sub = panel[panel.iso3 == c].set_index("year").sort_index()
    if "v2clfmove" in sub.columns and "SM.POP.NETM" in sub.columns:
        real_pairs_exit[c] = (sub["v2clfmove"], sub["SM.POP.NETM"])
result_exit = panel_granger(real_pairs_exit, lag=1, seed=42)
print(result_exit)

# --- SECONDARY application (kept from V6): corruption -> remittances ---
print("\n=== SECONDARY -- Control of Corruption -> Remittances (% GDP), pooled across countries ===")
real_pairs = {}
for c in COUNTRIES:
    sub = panel[panel.iso3 == c].set_index("year").sort_index()
    if "GOV_WGI_CC" in sub.columns and "BX.TRF.PWKR.DT.GD.ZS" in sub.columns:
        real_pairs[c] = (sub["GOV_WGI_CC"], sub["BX.TRF.PWKR.DT.GD.ZS"])
result = panel_granger(real_pairs, lag=1, seed=42)
print(result)
print("\nInterpretation reminder: even pooled across 6 countries, both tests are precedence checks")
print("between NATIONAL aggregates, not a test of individual perception -> migration intent.")

## 17 · Basic forecasting models — paper-sync

**Where this goes:** Section 5.1/5.4 ("remittance economy... masks structural
weaknesses"). **Not** a forward-looking claim anywhere in the paper — no section
argues what happens in 2029, so the point of this section is the **historical fit
quality** (does the series trend cleanly or not), not the forecast itself.

Three methods (linear trend, Holt, small-grid ARIMA) are kept side by side —
this is existing, already-validated work (each is backtested against the last 5
held-out years, Section 19/20 below), and multi-method agreement is itself a
robustness argument worth keeping. What changes in V7 is the **framing**: the
number to quote in the paper is the linear-trend R² (a measure of how cleanly the
series trends historically), not the 2029 endpoint.

**Suggested paper text:**
> Nepal's remittances (% of GDP) show a strong, well-fit upward historical trend
> (linear R² = [X]), while Government Effectiveness (WGI) over the same period
> shows essentially no trend (linear R² = [Y]). This divergence — one series moving
> cleanly, the other going nowhere — is quantitative support for Section 5.4's
> argument that remittance inflows have not been accompanied by, and may actively
> mask, improvement in governance capacity.

With ~20–30 annual points, the sensible use of the forward projection itself is a
**short horizon** (5 years) shown mainly to illustrate how quickly the interval
widens — informative for a methods/limitations note, not a substantive claim.

In [ ]:
# 18 · Forecasting functions
def linear_trend_forecast(series, horizon=5, alpha=0.05):
    s = series.dropna()
    years = s.index.values.astype(float)
    X = sm.add_constant(years)
    model = sm.OLS(s.values, X).fit()
    future_years = np.arange(years.max() + 1, years.max() + horizon + 1)
    pred = model.get_prediction(sm.add_constant(future_years))
    frame = pred.summary_frame(alpha=alpha)
    return pd.DataFrame({"forecast": frame["mean"].values,
                          "lower": frame["obs_ci_lower"].values,
                          "upper": frame["obs_ci_upper"].values},
                         index=future_years.astype(int)), model

def holt_forecast(series, horizon=5):
    s = series.dropna()
    fit = Holt(s.values, initialization_method="estimated").fit(optimized=True)
    fc = fit.forecast(horizon)
    future_years = np.arange(s.index.max() + 1, s.index.max() + horizon + 1)
    return pd.Series(fc, index=future_years, name="holt_forecast"), fit

def small_grid_arima_forecast(series, horizon=5, alpha=0.05, max_p=2, max_d=1, max_q=2):
    s = series.dropna()
    best_aic, best_order, best_fit = np.inf, None, None
    for p in range(max_p + 1):
        for d in range(max_d + 1):
            for q in range(max_q + 1):
                if p == 0 and q == 0:
                    continue
                try:
                    fit = ARIMA(s.values, order=(p, d, q)).fit()
                    if fit.aic < best_aic:
                        best_aic, best_order, best_fit = fit.aic, (p, d, q), fit
                except Exception:
                    continue
    if best_fit is None:
        raise RuntimeError("No ARIMA order converged for this series.")
    fc = best_fit.get_forecast(steps=horizon)
    frame = fc.summary_frame(alpha=alpha)
    future_years = np.arange(s.index.max() + 1, s.index.max() + horizon + 1)
    return pd.DataFrame({"forecast": frame["mean"].values,
                          "lower": frame["mean_ci_lower"].values,
                          "upper": frame["mean_ci_upper"].values},
                         index=future_years.astype(int)), best_order

def plot_forecast_fan(history, forecasts_dict, ylabel="", title=None, savepath=None):
    fig, ax = plt.subplots(figsize=(7.5, 4.4))
    ax.plot(history.index, history.values, color="#222222", linewidth=1.8, label="Observed", zorder=5)
    palette = ["#B23A48", "#2A6F77", "#C98A2C"]
    for (name, fc), color in zip(forecasts_dict.items(), palette):
        if hasattr(fc, "columns") and "forecast" in fc.columns:
            ax.plot(fc.index, fc["forecast"], color=color, linewidth=2.0, linestyle="--", label=f"{name} forecast")
            ax.fill_between(fc.index, fc["lower"], fc["upper"], color=color, alpha=0.15)
        else:
            ax.plot(fc.index, fc.values, color=color, linewidth=2.0, linestyle="--", label=f"{name} forecast")
    ax.axvline(history.index.max(), color="#999999", linewidth=0.8, linestyle=":")
    ax.set_ylabel(ylabel)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True, nbins=8))
    ax.legend(frameon=False, fontsize=9, loc="best")
    if title:
        ax.set_title(title, fontsize=12, loc="left", fontweight="bold")
    fig.tight_layout()
    if savepath:
        fig.savefig(savepath, bbox_inches="tight")
        fig.savefig(savepath.rsplit(".", 1)[0] + ".pdf", bbox_inches="tight")
    return fig

print("Forecasting functions ready.")


def backtest(series, holdout=5, alpha=0.05):
    """Fit each model through year T-holdout, forecast `holdout` years ahead, and
    check against the actual held-out values -- this is what makes the R^2/ARIMA
    numbers defensible as a historical-fit claim rather than just an in-sample fit."""
    s = series.dropna().sort_index()
    if len(s) < holdout + 8:
        print(f"  (only {len(s)} observations -- too few to hold out {holdout} years "
              "and still fit sensibly; backtest skipped)")
        return None
    train, test = s.iloc[:-holdout], s.iloc[-holdout:]
    rows = []
    lt, _ = linear_trend_forecast(train, horizon=holdout, alpha=alpha)
    hw, _ = holt_forecast(train, horizon=holdout)
    ar, order = small_grid_arima_forecast(train, horizon=holdout, alpha=alpha)
    for name, fc in [("Linear trend", lt), ("Holt", hw), (f"ARIMA{order}", ar)]:
        for yr, actual in test.items():
            if yr not in fc.index:
                continue
            point = fc.loc[yr, "forecast"] if hasattr(fc, "columns") else fc.loc[yr]
            has_ci = hasattr(fc, "columns") and "lower" in fc.columns
            in_band = float(fc.loc[yr, "lower"] <= actual <= fc.loc[yr, "upper"]) if has_ci else np.nan
            ape = abs((actual - point) / actual) * 100 if actual != 0 else np.nan
            rows.append({"model": name, "year": yr, "actual": actual, "forecast": point,
                         "in_95pct_band": in_band, "abs_pct_error": ape})
    detail = pd.DataFrame(rows)
    summary = detail.groupby("model").agg(
        mean_abs_pct_error=("abs_pct_error", "mean"),
        coverage=("in_95pct_band", "mean"),
        n_years_tested=("year", "count"),
    ).round(3)
    return detail, summary

print("backtest() ready.")

In [ ]:
# 19 · Figure E -- Nepal remittances (% GDP): observed + 3 forecasts
hist = nepal["BX.TRF.PWKR.DT.GD.ZS"].dropna()
lt, lt_model = linear_trend_forecast(hist, horizon=5)
hw, hw_fit = holt_forecast(hist, horizon=5)
ar, ar_order = small_grid_arima_forecast(hist, horizon=5)

print(f"Linear trend R^2 = {lt_model.rsquared:.3f} | Best ARIMA order = {ar_order}")
print("(R^2 is the number to quote in the paper -- see the markdown note above on why)")

fig_e = plot_forecast_fan(hist, {"Linear trend": lt, "Holt": hw, f"ARIMA{ar_order}": ar},
                           ylabel="Remittances Received (% of GDP)",
                           title="Nepal: Remittances (% of GDP) — Observed & 5-Year Forecast",
                           savepath=f"{FIG_DIR}/fig_e_remittances_forecast.png")
plt.show()

print("\n--- Backtest: fit through 2019, forecast 2020-2024, compare to what actually happened ---")
bt = backtest(hist, holdout=5)
if bt is not None:
    detail_e, summary_e = bt
    display(summary_e)
    print("\ncoverage = fraction of the 5 held-out years whose ACTUAL value fell inside that ")
    print("model's own 95% forecast band; mean_abs_pct_error = average |actual-forecast|/actual.")

In [ ]:
# 20 · Figure F -- Nepal Government Effectiveness (WGI): observed + 3 forecasts
hist_ge = nepal["GOV_WGI_GE"].dropna()
lt_ge, lt_ge_model = linear_trend_forecast(hist_ge, horizon=5)
hw_ge, _ = holt_forecast(hist_ge, horizon=5)
ar_ge, ar_ge_order = small_grid_arima_forecast(hist_ge, horizon=5)

print(f"Linear trend R^2 = {lt_ge_model.rsquared:.3f} | Best ARIMA order = {ar_ge_order}")
print("(R^2 is the number to quote in the paper -- see the markdown note above on why)")

fig_f = plot_forecast_fan(hist_ge, {"Linear trend": lt_ge, "Holt": hw_ge, f"ARIMA{ar_ge_order}": ar_ge},
                           ylabel="Government Effectiveness (WGI estimate, ~ -2.5 to 2.5)",
                           title="Nepal: Government Effectiveness — Observed & 5-Year Forecast",
                           savepath=f"{FIG_DIR}/fig_f_governance_forecast.png")
plt.show()

print("\n--- Backtest: fit through 2019, forecast 2020-2024, compare to what actually happened ---")
bt_ge = backtest(hist_ge, holdout=5)
if bt_ge is not None:
    detail_f, summary_f = bt_ge
    display(summary_f)

**Interpretation note (fill in the bracketed values once run, then use directly in
Section 5.1/5.4):**

> The two trends diverge sharply: remittances (% GDP) fit a linear trend with
> R² = [0.845 in the prior run of this notebook — reconfirm], while Government
> Effectiveness fits at R² = [0.074 in the prior run — reconfirm]. In plain terms,
> remittance dependence has risen on a clear, predictable path over the observed
> period, while the underlying governance capacity it is often assumed to reflect
> has not moved in any consistent direction. This is quantitative support — not
> proof — for treating the "remittance economy" as something that coexists with,
> rather than resolves, the implementation gap documented qualitatively in
> Sections 5.1–5.2.

## 21 · Mediation — the one real individual-level test (paper-sync overview)

**Where this goes:** Section 3.3/3.4 (the theoretical core) and Section 8's own
template for calibrated "consistent with" language.

This is the actual test of mediation the theory calls for — a confirmatory factor
analysis on a real trust battery, then a path model with a bootstrapped indirect
effect — written and mechanically verified (fit converges, correctly recovers a
known indirect effect on synthetic data with a planted mediation structure).

**Real data since integrated:** Asian Barometer Wave 1 (2005) and Wave 2 (2013), 5
of 6 countries (no Bhutan). Sections 21c onward load it and actually call
`run_mediation()` on real respondents. **V7 changes to what's run, not to the model
itself** (see Section 21n for the specifics): (1) Nepal-only, in addition to
pooled-5-country, since Nepal is the paper's actual focal case; (2) satisfaction
with democracy as a second outcome alongside regime preference, so the result can
be triangulated across two related-but-distinct attitude measures instead of resting
on one.

`country ~ district` nesting (a genuinely multilevel model) is the one piece
`semopy` doesn't do natively — Python's SEM tooling isn't at full parity with R's
`lavaan` here. Not attempted in either V6 or V7; flagged again at the point where it
matters (Section 21n).

In [ ]:
# 21b · CFA -> mediation -> bootstrapped indirect effect
import semopy

def build_mediation_spec(trust_items, condition_col="objective_condition", outcome_col="migration_outcome"):
    """lavaan-style model description: Trust as a latent variable measured by
    `trust_items`, mediating between `condition_col` and `outcome_col`."""
    loadings = " + ".join(trust_items)
    return f"""
Trust =~ {loadings}
{outcome_col} ~ Trust + {condition_col}
Trust ~ {condition_col}
"""

def run_mediation(df, trust_items, condition_col="objective_condition",
                   outcome_col="migration_outcome", n_boot=1000, seed=0):
    """Fits the CFA + path model, then bootstraps the indirect effect
    (condition -> Trust -> outcome) for a percentile CI."""
    required = list(trust_items) + [condition_col, outcome_col]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"mediation_df is missing required column(s): {missing}")
    if len(df.dropna(subset=required)) < 50:
        raise ValueError(
            f"Only {len(df.dropna(subset=required))} complete rows -- too few for a "
            "stable CFA + path model."
        )

    desc = build_mediation_spec(trust_items, condition_col, outcome_col)
    model = semopy.Model(desc)
    model.fit(df)
    params = model.inspect()

    rng = np.random.default_rng(seed)
    boots, n = [], len(df)
    for _ in range(n_boot):
        sample = df.iloc[rng.integers(0, n, n)].reset_index(drop=True)
        try:
            m = semopy.Model(desc)
            m.fit(sample)
            ins = m.inspect()
            a = ins[(ins["lval"] == "Trust") & (ins["rval"] == condition_col)]["Estimate"].values[0]
            b = ins[(ins["lval"] == outcome_col) & (ins["rval"] == "Trust")]["Estimate"].values[0]
            boots.append(a * b)
        except Exception:
            continue
    boots = np.array(boots)
    indirect_summary = {
        "indirect_effect_mean": float(boots.mean()) if len(boots) else np.nan,
        "ci_2.5": float(np.percentile(boots, 2.5)) if len(boots) else np.nan,
        "ci_97.5": float(np.percentile(boots, 97.5)) if len(boots) else np.nan,
        "n_successful_bootstraps": len(boots),
        "boot_samples": boots,
        "significant": bool(len(boots) and not (np.percentile(boots, 2.5) < 0 < np.percentile(boots, 97.5))),
    }
    return params, indirect_summary

print("run_mediation() defined and ready.")

## 21c · Real ABS data: loading, harmonization, and what it changes

**What's genuinely new:** Wave 1 (2005) and Wave 2 (2013) are household-survey
microdata with the multi-item trust battery the CFA needs, for 5 of our 6
countries (Nepal, India, Bangladesh, Sri Lanka, Pakistan -- no Bhutan). Column-to-
construct mappings below (Section 21e) are confirmed against the actual ABS
questionnaires for both waves, not inferred from column patterns.

**Two things this does NOT change:**

1. **This is not a time series, and nothing below forecasts anything.** Two waves,
   8 years apart, is 2 data points. What follows is a **before/after comparison**
   (2005 vs. 2013), a legitimate and different claim from "the trend continuing
   forward."
2. **Wave 1 and Wave 2 are independent cross-sections, not a panel.** The mediation
   model below is run **separately per wave**, partly because there's no way to
   track an individual's change over time, and partly because the two waves' trust
   batteries have different item counts (9 vs. 12).

**Migration/emigration intention: confirmed absent from both instruments.** Both
full questionnaires (2005 Set A/B, 2013 Set A) were checked line by line, and
neither contains a migration- or emigration-intention item. `GBC32` (Wave 2), an
earlier candidate based only on its 4-point likelihood scale, is ruled out by the
actual question text -- it asks how likely a respondent thinks it is that
government solves the problem *they themselves named* as the country's biggest,
within 5 years (Q31/Q32) -- a genuine government-responsiveness-**expectation**
measure (Section 21m gives this its own paper-sync note, tied to Endsley's
"projection" stage), not a migration proxy. The mediation model uses **regime
preference** and, new in V7, **satisfaction with democracy** as outcomes instead --
real, defensible, but narrower claims than "migration intentions increase."

In [ ]:
# 21d · Text-response recoding infrastructure -- one set of rules that works
# for BOTH waves' response formats.
import re

def _strip_leading_code(text):
    if not isinstance(text, str):
        return text
    m = re.match(r'^\s*-?\d+\s*:\s*(.*)$', text)
    return m.group(1).strip() if m else text.strip()

MISSING_SUBSTRINGS = ["d.k", "don't know", "can't say", "no opinion", "n.a.",
                      "no response", "not asked", "refused", "missing"]

def _is_missing(stripped_text):
    if stripped_text is None:
        return True
    t = str(stripped_text).strip().lower()
    if t in ("-1", "nan", ""):
        return True
    return any(p in t for p in MISSING_SUBSTRINGS)

def recode_scale(series, mapping, name="", verbose=True):
    """mapping: {lowercase substring: ordinal value}, matched longest-pattern-
    first. Unmatched non-missing text -> NaN WITH a printed warning."""
    patterns = sorted(mapping.items(), key=lambda kv: -len(kv[0]))
    out, unmatched = [], set()
    for raw in series:
        stripped = _strip_leading_code(raw)
        if _is_missing(stripped):
            out.append(np.nan)
            continue
        low = str(stripped).strip().lower()
        hit = None
        for pat, val in patterns:
            if pat in low:
                hit = val
                break
        if hit is None:
            unmatched.add(stripped)
            out.append(np.nan)
        else:
            out.append(hit)
    if unmatched and verbose:
        print(f"  NOTE ({name}): {len(unmatched)} unmapped value(s) -> coded missing. "
              f"Add to the mapping below if these are substantive, not just noise: {sorted(unmatched)}")
    return pd.Series(out, index=series.index, dtype=float)

# ---- reusable scale dictionaries (each: text pattern -> ordinal value) ----
TRUST_SCALE = {"great deal": 4, "quite a lot": 3, "some": 3,
               "not very much": 2, "not at all": 1, "none at all": 1}
AGREE_SCALE = {"strongly disagree": 1, "strongly agree": 4, "disagree": 2, "agree": 3}
APPROVE_SCALE = {"strongly disapprove": 1, "strongly approve": 4, "disapprove": 2, "approve": 3}
CONDITION_LEVEL_SCALE = {"very bad": 1, "bad": 2, "so so": 3, "good": 4, "very good": 5}
CHANGE_SCALE = {"much worse": 1, "a little worse": 2, "worse": 2, "same": 3, "about the same": 3,
                "a little better": 4, "better": 4, "much better": 5}
SATISFACTION_SCALE = {"not at all satisfied": 1, "not very satisfied": 2,
                      "somewhat satisfied": 3, "fairly satisfied": 3,
                      "very satisfied": 4, "satisfied": 3,
                      "neither satisfied nor dissatisfied": 2.5}
LIKELY_SCALE = {"not at all likely": 1, "not very likely": 2, "likely": 3, "very likely": 4}
REGIME_PREF_SCALE = {
    "dictatorship is preferable": 1, "authoritarian government can": 1,
    "does not matter": 2, "doesn't matter": 2,
    "democracy is preferable": 3, "democracy is always preferable": 3,
}
print("recode_core.py loaded OK")

In [ ]:
# 21e · Column-role maps, Wave 1 (2005) and Wave 2 (2013)
# ---- Column-role maps -- confirmed against the actual ABS questionnaires ----
WAVE1_COUNTRY_COL = "v1"
WAVE1_TRUST_ITEMS = ["c13a", "c13c", "c13d", "c13e", "c13f", "c13g", "c13h", "c13i", "c13j"]
WAVE1_TRUST_LABELS = {
    "c13a": "Central/National government", "c13c": "Local government",
    "c13d": "Civil service", "c13e": "Police", "c13f": "Army",
    "c13g": "Courts", "c13h": "Parliament", "c13i": "Political parties",
    "c13j": "Election Commission",
}
WAVE1_CONDITION_ITEMS = {
    "b2": SATISFACTION_SCALE, "b7": SATISFACTION_SCALE,
    "b3": CHANGE_SCALE, "b4": CHANGE_SCALE, "b8": CHANGE_SCALE, "b9": CHANGE_SCALE,
}
WAVE1_REGIME_PREF_COL = "c23"
WAVE1_SATISFACTION_DEMOCRACY_COL = "c12"
WAVE1_PROBLEM_SOLVING_EXPECTATION_COL = None  # no Wave 1 equivalent of Wave 2's GBC32

WAVE2_COUNTRY_COL = "ccode"
WAVE2_TRUST_ITEMS = ["GB7a", "GB7b", "GB7c", "GB7e", "GB7f", "GB7g",
                     "GB7h", "GB7i", "GB7j", "GB7k", "GB7l", "GB7m"]
WAVE2_TRUST_LABELS = {
    "GB7a": "President", "GB7b": "Prime Minister",
    "GB7c": "National government (capital)", "GB7e": "Parliament",
    "GB7f": "Local government", "GB7g": "Courts", "GB7h": "Civil service",
    "GB7i": "Political parties", "GB7j": "Military/armed forces",
    "GB7k": "Police", "GB7l": "Newspapers", "GB7m": "Television",
}
WAVE2_CONDITION_ITEMS = {
    "GB1": CONDITION_LEVEL_SCALE, "GBC4": CONDITION_LEVEL_SCALE,
    "GB2": CHANGE_SCALE, "GB3": CHANGE_SCALE, "GBC5": CHANGE_SCALE, "GBC6": CHANGE_SCALE,
}
WAVE2_REGIME_PREF_COL = "GBC34"
WAVE2_SATISFACTION_DEMOCRACY_COL = "GBC26"
WAVE2_PROBLEM_SOLVING_EXPECTATION_COL = "GBC32"

# V7: no Voice/protest column is added here -- see Section 21e-extra below for
# why (this notebook does not guess at a column it can't verify against your
# questionnaire), and how to add one if your files have it.

print(f"Wave 1: {len(WAVE1_TRUST_ITEMS)} trust items | Wave 2: {len(WAVE2_TRUST_ITEMS)} trust items configured.")

In [ ]:
# 21f · ABS toolkit: loader, reliability check, and plotting helpers
def load_abs_wave(path, wave_label, year, country_col, trust_items, condition_items,
                   regime_pref_col, satisfaction_col, extra_item_col=None, verbose=True):
    """Loads one ABS wave (.xlsx), recodes every item this notebook uses onto a
    common numeric scale, and returns one row per respondent."""
    raw = pd.read_excel(path)
    if verbose:
        print(f"[{wave_label}] loaded {raw.shape[0]:,} rows x {raw.shape[1]} columns from {path}")

    expected = [country_col, regime_pref_col, satisfaction_col] + trust_items + list(condition_items)
    missing_cols = [c for c in expected if c not in raw.columns]
    if missing_cols and verbose:
        print(f"[{wave_label}] WARNING: expected column(s) not in this file, skipped: {missing_cols}")

    out = pd.DataFrame(index=raw.index)
    out["country"] = raw[country_col].apply(_strip_leading_code) if country_col in raw.columns else np.nan
    out["wave"] = wave_label
    out["year"] = year

    present_trust = [c for c in trust_items if c in raw.columns]
    for i, c in enumerate(present_trust):
        out[f"trust_item_{i}"] = recode_scale(raw[c], TRUST_SCALE, name=f"{wave_label}:{c}", verbose=verbose)
    trust_cols = [f"trust_item_{i}" for i in range(len(present_trust))]
    out["trust_composite"] = out[trust_cols].mean(axis=1, skipna=True) if trust_cols else np.nan
    out["n_trust_items_answered"] = out[trust_cols].notna().sum(axis=1) if trust_cols else 0

    cond_cols = []
    for c, scale in condition_items.items():
        if c in raw.columns:
            colname = f"_cond_{c}"
            out[colname] = recode_scale(raw[c], scale, name=f"{wave_label}:{c}", verbose=verbose)
            cond_cols.append(colname)
    if cond_cols:
        z = out[cond_cols].apply(lambda s: (s - s.mean()) / s.std(ddof=1) if s.std(ddof=1) else s * 0)
        out["condition_composite"] = z.mean(axis=1, skipna=True)
    else:
        out["condition_composite"] = np.nan

    if regime_pref_col in raw.columns:
        out["regime_preference"] = recode_scale(raw[regime_pref_col], REGIME_PREF_SCALE,
                                                 name=f"{wave_label}:{regime_pref_col}", verbose=verbose)
    if satisfaction_col in raw.columns:
        out["satisfaction_democracy"] = recode_scale(raw[satisfaction_col], SATISFACTION_SCALE,
                                                       name=f"{wave_label}:{satisfaction_col}", verbose=verbose)
    if extra_item_col and extra_item_col in raw.columns:
        out["extra_item_raw"] = recode_scale(raw[extra_item_col], LIKELY_SCALE,
                                              name=f"{wave_label}:{extra_item_col}", verbose=verbose)

    if verbose:
        print(f"[{wave_label}] countries: {sorted(out['country'].dropna().unique())}")
        print(f"[{wave_label}] trust_composite: mean={out['trust_composite'].mean():.2f} "
              f"(n valid={out['trust_composite'].notna().sum()}/{len(out)})")
    return out


def cronbach_alpha(item_df):
    """Cronbach's alpha for a set of Likert-type items."""
    item_df = item_df.dropna()
    k = item_df.shape[1]
    if k < 2 or len(item_df) < 3:
        return np.nan
    item_vars = item_df.var(axis=0, ddof=1)
    total_var = item_df.sum(axis=1).var(ddof=1)
    return (k / (k - 1)) * (1 - item_vars.sum() / total_var)


WAVE_COLORS = {"2005": "#9AA5B1", "2013": "#B23A48"}  # muted past vs. accent present


def plot_wave_comparison(df, value_col, ylabel, title, country_order=None, savepath=None):
    """Grouped bar chart: one group per country, one bar per wave. A BEFORE/AFTER
    comparison, not a trend line (Section 21c explains why 2 waves can't forecast)."""
    agg = df.groupby(["country", "wave"])[value_col].mean().unstack()
    if country_order:
        agg = agg.reindex(country_order)
    waves = list(agg.columns)
    x = np.arange(len(agg.index))
    width = 0.8 / len(waves)
    fig, ax = plt.subplots(figsize=(8, 4.5))
    for i, wave in enumerate(waves):
        offset = (i - (len(waves) - 1) / 2) * width
        ax.bar(x + offset, agg[wave].values, width=width,
               color=WAVE_COLORS.get(str(wave), "#999999"), label=wave)
    ax.set_xticks(x); ax.set_xticklabels(agg.index, rotation=0)
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=12, loc="left", fontweight="bold")
    ax.legend(frameon=False, title="Wave")
    fig.tight_layout()
    if savepath:
        fig.savefig(savepath, bbox_inches="tight")
        fig.savefig(savepath.rsplit(".", 1)[0] + ".pdf", bbox_inches="tight")
    return fig, agg


def plot_trust_by_institution(df, trust_cols, labels, wave_label, title_suffix="", savepath=None):
    """Horizontal bar chart: mean trust per institution (not the aggregate
    composite), for ONE wave, ranked high-to-low."""
    means = df[trust_cols].mean().rename(index=labels).sort_values()
    fig, ax = plt.subplots(figsize=(7, 0.4 * len(means) + 1.2))
    ax.barh(means.index, means.values, color="#5B6EE1")
    ax.set_xlabel("Mean trust (1-4 scale)")
    ax.set_title(f"Institutional trust, {wave_label}{title_suffix}", fontsize=11, loc="left", fontweight="bold")
    ax.set_xlim(1, 4)
    fig.tight_layout()
    if savepath:
        fig.savefig(savepath, bbox_inches="tight")
        fig.savefig(savepath.rsplit(".", 1)[0] + ".pdf", bbox_inches="tight")
    return fig, means


def plot_mediation_result(params, indirect, condition_col, outcome_col,
                           condition_label=None, outcome_label=None,
                           title_suffix="", savepath=None):
    """Two panels: (1) a hand-drawn path diagram with the three key structural
    coefficients, and (2) the bootstrap distribution of the indirect effect a*b.

    V7 fix: condition_col/outcome_col are the REAL column names used to fit the
    model (needed to look up coefficients in `params`); condition_label/
    outcome_label are what actually gets drawn in the boxes. V6 conflated these,
    so the diagram displayed internal variable names ("objective_condition",
    "migration_outcome") instead of readable text -- the same underlying issue
    as a raw indicator code, just in a different spot."""
    condition_label = condition_label or condition_col
    outcome_label = outcome_label or outcome_col

    def _coef(lval, rval):
        row = params[(params["lval"] == lval) & (params["rval"] == rval) & (params["op"] == "~")]
        return float(row["Estimate"].values[0]) if len(row) else None

    a = _coef("Trust", condition_col)
    b = _coef(outcome_col, "Trust")
    c = _coef(outcome_col, condition_col)

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
    ax = axes[0]
    ax.set_xlim(0, 10); ax.set_ylim(0, 6); ax.axis("off")
    boxes = {"condition": (1.3, 3), "Trust": (5, 5), "outcome": (8.7, 3)}
    labels = {"condition": condition_label, "Trust": "Trust\n(latent)", "outcome": outcome_label}
    for key, (x, y) in boxes.items():
        ax.add_patch(plt.Rectangle((x - 1.3, y - 0.6), 2.6, 1.2, fill=True,
                                    facecolor="#EDEFF1", edgecolor="#333333", zorder=2))
        ax.text(x, y, labels[key], ha="center", va="center", fontsize=8.5, zorder=3)

    def arrow(p1, p2, label):
        ax.annotate("", xy=p2, xytext=p1, arrowprops=dict(arrowstyle="->", color="#333333", lw=1.6))
        mx, my = (p1[0] + p2[0]) / 2, (p1[1] + p2[1]) / 2
        ax.text(mx, my + 0.28, label, ha="center", fontsize=9, color="#B23A48", fontweight="bold")

    arrow((2.3, 3.5), (4, 4.6), f"a = {a:.2f}" if a is not None else "a = ?")
    arrow((6, 4.6), (7.7, 3.5), f"b = {b:.2f}" if b is not None else "b = ?")
    arrow((2.3, 2.7), (7.7, 2.7), f"c' = {c:.2f}" if c is not None else "c' = ?")
    ax.set_title(f"Path model{title_suffix}", fontsize=11, loc="left", fontweight="bold")

    ax2 = axes[1]
    boots = np.asarray(indirect.get("boot_samples", []))
    if len(boots):
        ax2.hist(boots, bins=30, color="#5B6EE1", alpha=0.75)
        ax2.axvline(0, color="#333333", linewidth=1, linestyle="--")
        ax2.axvline(indirect["ci_2.5"], color="#B23A48", linewidth=1.2, linestyle=":")
        ax2.axvline(indirect["ci_97.5"], color="#B23A48", linewidth=1.2, linestyle=":")
        ax2.set_title(f"Bootstrapped indirect effect (a x b), n={indirect['n_successful_bootstraps']}",
                      fontsize=11, loc="left", fontweight="bold")
        ax2.set_xlabel(f"mean={indirect['indirect_effect_mean']:.3f}  "
                       f"95% CI [{indirect['ci_2.5']:.3f}, {indirect['ci_97.5']:.3f}]  "
                       f"significant={indirect['significant']}", fontsize=8.5)
    else:
        ax2.text(0.5, 0.5, "No bootstrap samples to plot", ha="center", va="center")
        ax2.axis("off")
    fig.tight_layout()
    if savepath:
        fig.savefig(savepath, bbox_inches="tight")
        fig.savefig(savepath.rsplit(".", 1)[0] + ".pdf", bbox_inches="tight")
    return fig


def plot_abs_vs_expert_governance(abs_df, panel_df, expert_col, expert_label,
                                   wave_year_map=None, savepath=None):
    """Scatter: expert-coded governance score (WGI or V-Dem) vs. ABS aggregate
    institutional trust, one point per country x wave. Convergent-validity check."""
    wave_year_map = wave_year_map or {}
    abs_agg = (abs_df.groupby(["country", "wave"])["trust_composite"].mean()
                       .reset_index().rename(columns={"trust_composite": "abs_trust"}))
    abs_agg["year"] = abs_agg["wave"].map(lambda w: wave_year_map.get(w, w))
    abs_agg["year"] = pd.to_numeric(abs_agg["year"], errors="coerce")

    merged = abs_agg.merge(panel_df[["country", "year", expert_col]], on=["country", "year"], how="left")
    missing = merged[merged[expert_col].isna()]
    if len(missing) == len(merged):
        print("WARNING: no rows matched between ABS countries/years and the panel -- check that "
              "country spelling and the wave->year mapping line up.")
        return None, merged
    elif len(missing):
        print(f"NOTE: {len(missing)}/{len(merged)} country-wave rows had no matching {expert_col} "
              "value in the panel (dropped from the plot).")
    merged = merged.dropna(subset=[expert_col])

    fig, ax = plt.subplots(figsize=(6.5, 5.2))
    for wave, g in merged.groupby("wave"):
        ax.scatter(g[expert_col], g["abs_trust"], s=70,
                   color=WAVE_COLORS.get(str(wave), "#999999"), label=str(wave), zorder=3)
        for _, row in g.iterrows():
            ax.annotate(row["country"], (row[expert_col], row["abs_trust"]),
                        fontsize=7.5, xytext=(4, 4), textcoords="offset points")
    if len(merged) >= 3:
        r = merged[expert_col].corr(merged["abs_trust"])
        ax.set_title(f"ABS Aggregate Trust vs. {expert_label} (r={r:.2f}, n={len(merged)})",
                     fontsize=11, loc="left", fontweight="bold")
    else:
        ax.set_title(f"ABS Aggregate Trust vs. {expert_label}", fontsize=11, loc="left", fontweight="bold")
    ax.set_xlabel(expert_label); ax.set_ylabel("ABS Mean Institutional Trust (1-4)")
    ax.legend(frameon=False, title="Wave")
    fig.tight_layout()
    if savepath:
        fig.savefig(savepath, bbox_inches="tight")
        fig.savefig(savepath.rsplit(".", 1)[0] + ".pdf", bbox_inches="tight")
    return fig, merged


print("ABS toolkit ready: load_abs_wave(), cronbach_alpha(), plot_wave_comparison(), "
      "plot_trust_by_institution(), plot_mediation_result(), plot_abs_vs_expert_governance()")

In [ ]:
# 21g · Load both ABS waves, combine, and check the trust battery's reliability
try:
    from google.colab import files
    print("Select BOTH your Wave 1 (2005) and Wave 2 (2013) Asian Barometer .xlsx files "
          "(multi-select in the file picker):")
    uploaded = files.upload()
    abs_paths = list(uploaded.keys())
except ImportError:
    abs_paths = ["wave1_2005.xlsx", "wave2_2013.xlsx"]
    print(f"Not running in Colab -- looking for {abs_paths} in the working directory.")

wave1_path = wave2_path = None
for p in abs_paths:
    try:
        cols = pd.read_excel(p, nrows=0).columns
    except Exception as e:
        print(f"Could not read {p}: {e}")
        continue
    if WAVE1_COUNTRY_COL in cols and WAVE1_REGIME_PREF_COL in cols:
        wave1_path = p
    elif WAVE2_COUNTRY_COL in cols and WAVE2_REGIME_PREF_COL in cols:
        wave2_path = p
if wave1_path is None or wave2_path is None:
    print(f"NOTE: auto-detected wave1_path={wave1_path}, wave2_path={wave2_path} -- if either "
          "is None, set it manually and re-run this cell.")

w1_df = (load_abs_wave(wave1_path, "2005", 2005, WAVE1_COUNTRY_COL, WAVE1_TRUST_ITEMS,
                        WAVE1_CONDITION_ITEMS, WAVE1_REGIME_PREF_COL, WAVE1_SATISFACTION_DEMOCRACY_COL,
                        WAVE1_PROBLEM_SOLVING_EXPECTATION_COL) if wave1_path else None)
w2_df = (load_abs_wave(wave2_path, "2013", 2013, WAVE2_COUNTRY_COL, WAVE2_TRUST_ITEMS,
                        WAVE2_CONDITION_ITEMS, WAVE2_REGIME_PREF_COL, WAVE2_SATISFACTION_DEMOCRACY_COL,
                        WAVE2_PROBLEM_SOLVING_EXPECTATION_COL) if wave2_path else None)
abs_df = pd.concat([d for d in [w1_df, w2_df] if d is not None], ignore_index=True)

print(f"\nCombined ABS respondent-level data: {abs_df.shape[0]:,} respondents across "
      f"{abs_df['wave'].nunique()} wave(s); countries: {sorted(abs_df['country'].dropna().unique())}")

print("\nTrust-battery internal consistency (Cronbach's alpha) per wave:")
for wave, g in abs_df.groupby("wave"):
    trust_cols = [c for c in g.columns if c.startswith("trust_item_") and g[c].notna().sum() > 10]
    a = cronbach_alpha(g[trust_cols])
    flag = "" if (pd.isna(a) or a >= 0.7) else "  <-- below 0.7, treat the composite/CFA with caution"
    print(f"  wave {wave}: alpha={a:.3f} over {len(trust_cols)} items "
          f"(n={g[trust_cols].dropna().shape[0]} complete respondents){flag}")

### Trust figures (aggregate + institution-level) — paper-sync

**Where this goes:** Fig. 2's **"Public trust / institutional legitimacy"** node,
and Section 5.1/5.6 ("entrenched patronage," "growing crisis of trust").

**Suggested paper text:**
> Aggregate institutional trust [rose/fell] in Nepal between 2005 and 2013 (Figure
> [X]). Breaking trust down by institution (Figure [X+1]) shows [which institutions
> rank lowest/highest once run] — a pattern consistent with Section 5.1's argument
> that trust deficits concentrate in political/representative institutions
> specifically, rather than being uniform across the state.

**What it infers:** the institution-level breakdown (already built in this
notebook — `plot_trust_by_institution()`, called for both waves below) is what lets
the paper make a claim more specific than "trust is low" — e.g., whether trust in
political parties or parliament sits well below trust in police or courts, which is
the more precise empirical echo of Section 5.1's "entrenched patronage, weak
accountability" language than an aggregate number alone.

In [ ]:
# 21h · Figure -- institutional trust by country, 2005 vs. 2013
fig_abs_trust, agg_trust = plot_wave_comparison(
    abs_df, "trust_composite", "Mean Institutional Trust (1-4 scale)",
    "Institutional Trust by Country, 2005 vs. 2013 (Asian Barometer)",
    country_order=["Nepal", "India", "Bangladesh", "Sri Lanka", "Pakistan"],
    savepath=f"{FIG_DIR}/fig_abs_trust_wave.png")
plt.show()
display(agg_trust.round(2))

# Which institutions specifically, per wave -- nameable since the full
# questionnaires confirmed every trust-battery letter (Section 21e).
w1_only = abs_df[abs_df["wave"] == "2005"]
w1_trust_cols = [f"trust_item_{i}" for i in range(len(WAVE1_TRUST_ITEMS))]
fig_trust_inst_2005, means_2005 = plot_trust_by_institution(
    w1_only, w1_trust_cols, dict(zip(w1_trust_cols, [WAVE1_TRUST_LABELS[c] for c in WAVE1_TRUST_ITEMS])),
    "2005", savepath=f"{FIG_DIR}/fig_abs_trust_institution_2005.png")
plt.show()

w2_only = abs_df[abs_df["wave"] == "2013"]
w2_trust_cols = [f"trust_item_{i}" for i in range(len(WAVE2_TRUST_ITEMS))]
fig_trust_inst_2013, means_2013 = plot_trust_by_institution(
    w2_only, w2_trust_cols, dict(zip(w2_trust_cols, [WAVE2_TRUST_LABELS[c] for c in WAVE2_TRUST_ITEMS])),
    "2013", savepath=f"{FIG_DIR}/fig_abs_trust_institution_2013.png")
plt.show()

### Regime preference figure — paper-sync

**Where this goes:** Fig. 2's "voice vs. loyalty" node — **partially**. Regime
preference (support for democracy vs. an authoritarian alternative) is a
system-legitimacy attitude; it is *related to* but not literally "Voice" in
Hirschman's sense (protest, contacting officials, petitioning). Say this precisely
in the paper rather than treating the two as interchangeable — see Section
21e-extra below for how to add a literal Voice item if your files have one.

**Suggested paper text:**
> Support for democracy over an authoritarian alternative [rose/fell/stayed flat]
> in Nepal between 2005 and 2013 (Figure [X]) — [read alongside the trust and
> condition figures once run].

In [ ]:
# 21i · Figure -- regime preference (support for democracy) by country, 2005 vs. 2013
fig_abs_regime, agg_regime = plot_wave_comparison(
    abs_df, "regime_preference",
    "Mean Pro-Democracy Preference (1=Authoritarian-OK .. 3=Always-Prefer-Democracy)",
    "Regime Preference by Country, 2005 vs. 2013 (Asian Barometer)",
    country_order=["Nepal", "India", "Bangladesh", "Sri Lanka", "Pakistan"],
    savepath=f"{FIG_DIR}/fig_abs_regime_wave.png")
plt.show()
display(agg_regime.round(2))

### Perceived condition figure — paper-sync

**Where this goes:** **Definition 1.1** — this is the closest available
individual-level proxy for the *Perceived Opportunity Landscape* itself, since
Definition 1.1 explicitly describes the construct as mediating between objective
conditions and trust/migration outcomes, and this composite (self-rated current +
retrospective + prospective economic condition) sits in exactly that position in
the mediation model below (condition → Trust → outcome).

**Suggested paper text:**
> Perceived national/economic condition — a composite of current, retrospective,
> and prospective self-rated economic evaluations, and this notebook's closest
> operational proxy for the Perceived Opportunity Landscape (Definition 1.1) —
> [rose/fell] in Nepal between 2005 and 2013 (Figure [X]).

**Caveat to keep in the paper's own language:** this composite captures the
*content* of perceived condition, not the full perceive→comprehend→project
*process* Definition 1.1 describes. It is a defensible proxy for one input to that
process, not a direct measurement of situation awareness itself.

In [ ]:
# 21j · Figure -- perceived economic/national condition by country, 2005 vs. 2013
fig_abs_condition, agg_condition = plot_wave_comparison(
    abs_df, "condition_composite", "Mean Perceived Condition (z-scored composite; higher = better/improving)",
    "Perceived Conditions by Country, 2005 vs. 2013 (Asian Barometer)",
    country_order=["Nepal", "India", "Bangladesh", "Sri Lanka", "Pakistan"],
    savepath=f"{FIG_DIR}/fig_abs_condition_wave.png")
plt.show()
display(agg_condition.round(2))

### 21e-extra · OPTIONAL scaffold — a literal "Voice" item, if your files have one

**Not run automatically, and does not guess a column name.** Regime preference
(above) is a general system-legitimacy attitude — related to, but not literally,
"Voice" in Hirschman's sense (protest, contacting officials, petitioning,
signing petitions). If either ABS wave you already have includes a
political-participation/protest battery, the *same* `run_mediation()` pipeline
already built above can test it as a second, more precise Voice-side outcome — no
new data, just a different existing column, checked against your questionnaire the
same way `GBC32` was checked in Section 21c.

This cell just lists every raw column so you can scan for it yourself:

In [ ]:
# Run this AFTER Section 21g (needs wave1_path/wave2_path to already be set).
if "wave1_path" in dir() and wave1_path:
    try:
        raw_check = pd.read_excel(wave1_path, nrows=0)
        print("Wave 1 (2005) raw columns:")
        print(list(raw_check.columns))
    except Exception as e:
        print("Could not preview Wave 1 raw columns:", e)
else:
    print("Run Section 21g first, then re-run this cell.")

if "wave2_path" in dir() and wave2_path:
    try:
        raw_check2 = pd.read_excel(wave2_path, nrows=0)
        print("\nWave 2 (2013) raw columns:")
        print(list(raw_check2.columns))
    except Exception as e:
        print("Could not preview Wave 2 raw columns:", e)

# Once you've identified a real protest/participation item (confirmed against
# the actual question wording, not just guessed from the column name), wire it
# in the same way GBC32 was wired:
#   WAVE1_VOICE_COL = "cXX"   # <- replace with the real, confirmed column
#   WAVE2_VOICE_COL = "GBxx"  # <- replace with the real, confirmed column
# then reload via load_abs_wave(..., extra_item_col=WAVE1_VOICE_COL) the same
# way GBC32 is loaded below, and run:
#   run_mediation_for_subset(nepal_only, "extra_item_raw", "Nepal-only (Voice)")
# using the wrapper defined in Section 21n.

## Why there's no forecast figure in this ABS section (unlike Sections 17-20)

If the instinct at this point is "now project 2005→2013 forward the way Section
19/20 did" — don't. Those forecasts work because they had ~25-30 *annual* points; a
linear trend, Holt fit, or ARIMA model needs enough points to separate a real trend
from noise. **Two points define a line perfectly (R²=1) by construction, which is a
sign the fit is meaningless, not a sign it's good.** There is no statistically
defensible way to forecast an ABS-based series from Wave 1 + Wave 2 alone. If a
third wave becomes available later, revisit this.

### GBC32 (government problem-solving expectation) — paper-sync

**Where this goes:** Section 3.3 (Endsley's situation-awareness model) —
specifically, this is this notebook's closest empirical analog for the
**"projection"** stage. Endsley's model is perception → comprehension →
**projection** (the forward-looking inference from present state to expected
future state); `GBC32` asks respondents to do exactly that for a concrete problem
they themselves named, within a 5-year horizon.

**Suggested paper text:**
> As a Wave-2-only (2013) illustration of the "projection" stage in Endsley's model
> (Section 3.3), respondents' expectation that government would solve their
> self-identified top problem within five years [was low/moderate/high and
> varied by country as follows once run] (Figure [X]). This is not a migration
> item — the outcome is government-responsiveness expectation, confirmed against
> the actual question wording (Section 21c) — but it is a direct, if partial,
> individual-level operationalization of the projection mechanism the paper's
> theoretical framework rests on.

In [ ]:
# 21m · Figure -- GBC32, government problem-solving expectation (Wave 2 / 2013
# only). Confirmed via the actual questionnaire (Q31/Q32): respondents first
# named their country's biggest problem, then rated how likely they think it
# is the government solves it within 5 years. Not a migration item -- see 21c.
if "extra_item_raw" in abs_df.columns and abs_df["extra_item_raw"].notna().any():
    cand = abs_df[abs_df["wave"] == "2013"].groupby("country")["extra_item_raw"].agg(["mean", "count"])
    print("GBC32 by country (2013 only) -- 1=not at all likely government solves their "
          "self-identified top problem within 5 years .. 4=very likely:")
    display(cand.round(2))

    fig_cand, ax = plt.subplots(figsize=(6.5, 4))
    order = ["Nepal", "India", "Bangladesh", "Sri Lanka", "Pakistan"]
    vals = cand.reindex(order)["mean"]
    ax.bar(vals.index, vals.values, color="#C98A2C")
    ax.set_ylabel("Mean Likelihood Government Solves Top Problem (1-4)")
    ax.set_title("Government Problem-Solving Expectation by Country, 2013\n(the \"projection\" stage, Section 3.3)",
                 fontsize=11, loc="left", fontweight="bold")
    fig_cand.tight_layout()
    fig_cand.savefig(f"{FIG_DIR}/fig_abs_problem_solving_expectation.png", bbox_inches="tight")
    plt.show()
else:
    print("extra_item_raw not present/all-missing -- nothing to plot.")

## 21n · The mediation model — paper-sync

**Where this goes:** Section 3.3/3.4 (the theoretical core) and Section 8's
template for calibrated language — this is the strongest, most defensible result
this notebook produces.

**What changed in V7 (see Section 21 overview above for the full rationale):**

1. **Nepal-only, in addition to pooled-5-country.** RQ1–3 are explicitly about
   Nepal; pooling all 5 ABS countries together is really an RQ4 regional check, not
   the paper's headline claim. Both are run below — Nepal-only is now primary.
2. **Two outcomes: regime preference (as in V6) and satisfaction with democracy
   (new).** If the indirect effect replicates — same direction, both significant —
   across two related-but-distinct democracy-attitude outcomes, that is a real
   triangulation argument for "consistent with the hypothesized pathway," stronger
   than resting on one outcome. This also repurposes the data that V6 spent on a
   standalone satisfaction-with-democracy bar chart into an actual test, rather
   than a fourth descriptive figure.

**Not changed, and still the honest limits to state alongside any result below:**
the outcome is regime preference / satisfaction with democracy, **not** migration
intention (confirmed absent from both questionnaires, Section 21c); this is two
independent cross-sections, not a panel; pooling countries without a country
fixed-effect is a real simplification — if a reviewer pushes on this, country
dummies in the outcome/Trust equations, or separate per-country models, are the
next step, not attempted here; and `country ~ district` nesting is not attempted
in either version.

**Suggested paper text (template, calibrated — fill in once run):**
> In [wave year], for Nepal specifically, the bootstrapped indirect effect of
> perceived [condition] on [regime preference / support for democracy] through
> institutional trust was [statistically significant / not significant] and in the
> [predicted / unexpected] direction (indirect effect = [X], 95% CI [L, U]). This
> [replicated / did not replicate] in [the other wave], and [replicated / did not
> replicate] when satisfaction with democracy was used as an alternative outcome.
> Pooling across all five ABS-covered countries [confirms / does not confirm] the
> same pattern regionally (Section 6/RQ4).

Still not licensed by any of this: "proves," "confirms," or treating this as a test
of migration intentions specifically.

In [ ]:
# 21n (continued) · Run the REAL mediation model:
# Nepal-only (primary) + pooled-5-country (secondary/RQ4), each for BOTH
# regime_preference (as in V6) and satisfaction_democracy (new in V7).
OUTCOME_DISPLAY = {
    "regime_preference": "Regime Preference\n(Pro-Democracy Attitude)",
    "satisfaction_democracy": "Satisfaction with\nDemocracy",
}

def run_mediation_for_subset(df_subset, outcome_col, label):
    """Runs the CFA -> mediation -> bootstrap pipeline separately per wave on
    df_subset (already filtered to whichever countries you want), for one
    outcome column. Returns {wave: (params, indirect)}."""
    mediation_results = {}
    for wave, g in df_subset.groupby("wave"):
        trust_cols = [c for c in g.columns if c.startswith("trust_item_") and g[c].notna().sum() > 0]
        med_df = g[trust_cols + ["condition_composite", outcome_col]].rename(
            columns={"condition_composite": "objective_condition", outcome_col: "migration_outcome"})
        n_complete = len(med_df.dropna())
        print(f"=== {label} -- Wave {wave}, outcome={outcome_col}: "
              f"{n_complete} complete respondents across {len(trust_cols)} trust items ===")
        if n_complete < 50:
            print(f"  Skipped -- fewer than 50 complete rows.")
            continue
        params, indirect = run_mediation(med_df, trust_items=trust_cols,
                                          condition_col="objective_condition",
                                          outcome_col="migration_outcome", n_boot=1000, seed=0)
        mediation_results[wave] = (params, indirect)
        print(f"  indirect effect (perceived condition -> Trust -> {outcome_col}): "
              f"{indirect['indirect_effect_mean']:.3f}, 95% CI "
              f"[{indirect['ci_2.5']:.3f}, {indirect['ci_97.5']:.3f}], "
              f"significant={indirect['significant']}")
        plot_mediation_result(
            params, indirect, "objective_condition", "migration_outcome",
            condition_label="Perceived National\nCondition",
            outcome_label=OUTCOME_DISPLAY.get(outcome_col, outcome_col),
            title_suffix=f" -- {label}, Wave {wave}",
            savepath=f"{FIG_DIR}/fig_mediation_{label.lower().replace(' ', '_').replace('(','').replace(')','')}_{outcome_col}_{wave}.png")
        plt.show()
    if len(mediation_results) == 2:
        waves_sorted = sorted(mediation_results)
        same_sign = (np.sign(mediation_results[waves_sorted[0]][1]["indirect_effect_mean"]) ==
                     np.sign(mediation_results[waves_sorted[1]][1]["indirect_effect_mean"]))
        both_sig = all(mediation_results[w][1]["significant"] for w in waves_sorted)
        print(f"\n{label} ({outcome_col}) across-wave check: same direction in both waves = {same_sign}, "
              f"significant in both waves = {both_sig}.")
    return mediation_results

nepal_only = abs_df[abs_df["country"] == "Nepal"]

print("################ PRIMARY: Nepal only, outcome = regime preference ################")
mediation_nepal_regime = run_mediation_for_subset(nepal_only, "regime_preference", "Nepal-only")

print("\n################ ROBUSTNESS: Nepal only, outcome = satisfaction with democracy ################")
mediation_nepal_satisfaction = run_mediation_for_subset(nepal_only, "satisfaction_democracy", "Nepal-only")

print("\n################ SECONDARY / RQ4 comparative: pooled across 5 ABS countries, outcome = regime preference ################")
mediation_pooled_regime = run_mediation_for_subset(abs_df, "regime_preference", "Pooled-5-country")

print("\nReminder: pooling countries without a country fixed-effect is a simplification -- see the")
print("markdown note above for the next step if a reviewer pushes on this.")

## 21p · What this section actually licenses saying in the paper

The real ABS mediation result above is the **strongest** evidence this notebook
produces — an individual-level test, not an aggregate co-movement check. Calibrated
language for it is the template in Section 21n above. Still not licensed: "proves,"
"confirms," or treating this as a test of migration intentions specifically — both
questionnaires were checked directly and neither contains one, so the outcome is
regime preference / satisfaction with democracy, and no wording can close that gap.
It's also two independent cross-sections, not a longitudinal test of individual
change, and treating 4-point Likert items as continuous CFA indicators is a common
simplification — an ordinal/WLSMV estimator would be the more careful next step if
a reviewer pushes on this.

### Triangulation — paper-sync

**Where this goes:** Section 3.3 — this is what justifies *introducing* the
Perceived Opportunity Landscape as a distinct construct in the first place, rather
than just using WGI/V-Dem-style expert-coded aggregates directly.

**Suggested paper text:**
> ABS aggregate institutional trust and WGI Government Effectiveness [correlate
> strongly / diverge] across the country-waves examined (r = [X], Figure [Y]). [If
> they diverge:] This divergence between individually-reported trust and
> expert-coded governance quality is itself evidence that citizen perception is not
> simply a noisy read of official governance scores — supporting the paper's core
> claim that a distinct, perception-based construct is needed alongside standard
> governance indicators.

In [ ]:
# 21o · Triangulation -- does ABS's individual-level aggregate trust track the
# expert-coded governance panel from Sections 5-7 for the SAME country-years?
# A convergent-validity check, not a forecast and not a substitute for either one.
fig_triangulation, merged_check = plot_abs_vs_expert_governance(
    abs_df, panel, "GOV_WGI_GE", label_for("GOV_WGI_GE"),
    wave_year_map={"2005": 2005, "2013": 2013},
    savepath=f"{FIG_DIR}/fig_abs_vs_wgi_triangulation.png")
if fig_triangulation is not None:
    plt.show()
    display(merged_check.round(3))

In [ ]:
# 22 · Package everything for download
import shutil
zip_path = shutil.make_archive(f"{OUT_DIR}/nepal_south_asia_analysis", "zip", OUT_DIR)
print("Packaged ->", zip_path)

try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print("Not running in Colab (or download API unavailable) -- "
          f"grab the files directly from {FIG_DIR} and {DATA_DIR}.")

## 23 · Summary — what changed in V7, and what's still deliberately out

**Trimmed (Remove/trim):** V-Dem cut from 33 to 20 indicators (economic controls
group removed entirely; state capacity cut to 1 of 3; corruption cut to 1 of 3 —
Section 3); the unfinished UN DESA direct cross-check removed from Section 8;
the standalone satisfaction-with-democracy bar chart replaced by using it as a
second mediation outcome (Section 21n); forecast framing shifted from "2029
prediction" to "historical fit robustness" (Section 17).

**Added, from data already loaded (Add):** `v2x_regime` given its own figure
(Section 12c) as the paper's own operationalization of "democratic resilience";
`v2clfmove` ("exit") made the primary co-movement/Granger pairing instead of an
unused, loaded-but-idle variable (Sections 16b/16d); remittances re-captioned as
evidence of *loyalty* alongside *exit* (Section 10); institution-level trust
breakdown given its own paper-sync note (it was already being called — see the
correction in Section 21h's markdown); a Voice-item scaffold added without
fabricating a column (Section 21e-extra); mediation re-run Nepal-only (primary)
alongside pooled-5-country (secondary), across two outcomes instead of one
(Section 21n); every figure's axis/legend/tick text now shows real names
(`label_for()`, `country_name()`) instead of raw codes.

**Deliberately, plainly out of scope (Don't force):** RQ2 (core-periphery spatial
inequality) and brain circulation (return migration, diaspora investment) — see
Section 0b for exactly why, stated once at the top rather than caveated
figure-by-figure.

**Before anything here goes in the paper:** run this notebook top-to-bottom with
your real WGI/V-Dem/ABS files — nothing in this V7 rewrite has been executed; every
number, R², p-value, and figure above needs a fresh run to actually exist.